#Step 0

In [1]:
# ============================================================
# Notebook 2 — Cell 0: Setup + Load Master Transcripts
# ============================================================
# PURPOSE: Install Notebook 2 libraries, reload CONFIG and
# utilities from Drive, load the 3 master JSONs produced
# by Notebook 1.
#
# Libraries:
#   sentence-transformers — embed chunks as vectors (Step 6)
#   faiss-cpu             — fast similarity search (Step 6)
#   rank_bm25             — keyword search index (Step 6)
#   groq                  — LLM API for QA + summarization (Step 9)
#   rouge-score           — ROUGE evaluation (Step 10)
# ============================================================

import subprocess, sys

print("── Installing Notebook 2 libraries ─────────────────────")
packages = [
    'sentence-transformers',
    'faiss-cpu',
    'rank_bm25',
    'groq',
    'rouge-score',
    'numpy==1.26.4',   # pin for compatibility
]
for pkg in packages:
    r = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', pkg, '-q'],
        capture_output=True)
    print(f"  {'✅' if r.returncode == 0 else '❌'} {pkg}")

print()

# ── Mount Drive + load CONFIG ─────────────────────────────────
import os, json
from datetime import datetime
from google.colab import drive

drive.mount('/content/drive')
ROOT = '/content/drive/MyDrive/MeetingMind_v2'
with open(f'{ROOT}/config.json') as f:
    CONFIG = json.load(f)
print(f"✅ CONFIG loaded (version {CONFIG['version']})")

# ── Reload utilities ──────────────────────────────────────────
LOG_PATH = f"{CONFIG['logs_dir']}/run_log.txt"

def log(message, level='INFO'):
    timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    icons = {'INFO': '→', 'SUCCESS': '✅', 'WARNING': '⚠️', 'ERROR': '❌'}
    line = f"[{timestamp}] {icons.get(level,'→')} {message}"
    with open(LOG_PATH, 'a') as f:
        f.write(line + '\n')
    print(line)

def save_checkpoint(step_name, data):
    path = f"{CONFIG['checkpoints_dir']}/{step_name}.json"
    cp = {'step': step_name, 'saved_at': datetime.now().isoformat(),
          'version': CONFIG['version'], 'data': data}
    with open(path, 'w') as f:
        json.dump(cp, f, indent=2)
    log(f"Checkpoint saved: {step_name}", 'SUCCESS')

def load_checkpoint(step_name):
    path = f"{CONFIG['checkpoints_dir']}/{step_name}.json"
    if not os.path.exists(path):
        return None
    with open(path) as f:
        cp = json.load(f)
    log(f"Checkpoint loaded: {step_name}")
    return cp['data']

def print_report(title, metrics):
    width = 60
    print('=' * width)
    print(f"  {title}")
    print('=' * width)
    for label, value in metrics.items():
        print(f"  {label:<30} {value}")
    print('=' * width)

# ── Load all 3 master transcripts ────────────────────────────
print()
print("── Loading master transcripts from Drive ────────────────")

MASTERS = {}
for meeting_id in CONFIG['ami_meetings']:
    path = (f"{CONFIG['transcripts_dir']}/"
            f"{meeting_id}_master.json")
    if not os.path.exists(path):
        print(f"  ❌ MISSING: {meeting_id}_master.json")
        print(f"     Run Notebook 1 first to generate this file")
        continue
    with open(path) as f:
        MASTERS[meeting_id] = json.load(f)
    m = MASTERS[meeting_id]
    print(f"  ✅ {meeting_id}: "
          f"{m['total_turns']} turns  "
          f"{m['total_words']:,} words  "
          f"coverage={m['coverage_pct']}%")

print()

# ── Quick sanity check ────────────────────────────────────────
print("── Sanity check — sample turns ──────────────────────────")
for meeting_id, master in MASTERS.items():
    # Show first non-UNKNOWN, non-short turn
    good_turns = [t for t in master['turns']
                  if t['speaker'] != 'UNKNOWN'
                  and t['word_count'] >= 10]
    if good_turns:
        t = good_turns[0]
        preview = t['text'][:70] + ('...' if
                  len(t['text']) > 70 else '')
        print(f"  {meeting_id} [{t['start']:.1f}s] "
              f"{t['speaker']}: {preview}")

print()
print_report("Notebook 2 — Ready", {
    'Meetings loaded'    : str(len(MASTERS)),
    'Total turns'        : str(sum(m['total_turns']
                             for m in MASTERS.values())),
    'Total words'        : f"{sum(m['total_words'] for m in MASTERS.values()):,}",
    'Libraries'          : 'sentence-transformers, faiss-cpu, rank_bm25, groq',
    'GPU'                : 'CPU only — no heavy models in Notebook 2',
    'Next'               : 'Step 5 — meeting-aware chunking',
})

log("Notebook 2 Cell 0 complete — masters loaded", 'SUCCESS')

── Installing Notebook 2 libraries ─────────────────────
  ✅ sentence-transformers
  ✅ faiss-cpu
  ✅ rank_bm25
  ✅ groq
  ✅ rouge-score
  ✅ numpy==1.26.4

Mounted at /content/drive
✅ CONFIG loaded (version v2.0)

── Loading master transcripts from Drive ────────────────
  ✅ EN2001a: 786 turns  13,444 words  coverage=88.12%
  ✅ EN2001b: 513 turns  8,132 words  coverage=90.14%
  ✅ IS1009a: 139 turns  1,551 words  coverage=94.52%

── Sanity check — sample turns ──────────────────────────
  EN2001a [10.5s] SPEAKER_02: Does anyone want to see Steve's feedback from the specification? Is
  EN2001b [70.2s] SPEAKER_03: You see, I never get this. What's the point of... If this strap is sup...
  IS1009a [64.6s] SPEAKER_01: function is. So maybe we start with you. Yeah, my name is Fransena and...

  Notebook 2 — Ready
  Meetings loaded                3
  Total turns                    1438
  Total words                    23,127
  Libraries                      sentence-transformers, faiss-cpu, ra

#Step 5

In [2]:
# ============================================================
# Recovery cell — run after Runtime → Restart runtime
# ============================================================

import subprocess, sys

# Reinstall with correct numpy version
pkgs = ['sentence-transformers', 'faiss-cpu',
        'rank_bm25', 'groq', 'rouge-score']
for pkg in pkgs:
    subprocess.run([sys.executable, '-m', 'pip',
                    'install', pkg, '-q'], capture_output=True)

# Pin numpy AFTER other packages so it doesn't get overridden
subprocess.run([sys.executable, '-m', 'pip',
                'install', 'numpy==1.26.4', '-q'],
               capture_output=True)
print("✅ Libraries reinstalled")

✅ Libraries reinstalled


In [ ]:
# ============================================================
# Notebook 2 — ALWAYS RUN THIS FIRST after any session restart
# Cell 0: Install + fix numpy conflict + restart
# ============================================================

import subprocess, sys, os

print("Installing Notebook 2 libraries...")
pkgs = [
    'sentence-transformers',
    'faiss-cpu',
    'rank_bm25',
    'groq',
    'rouge-score',
    'google-generativeai',
]
for pkg in pkgs:
    r = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', pkg, '-q'],
        capture_output=True)
    print(f"  {'✅' if r.returncode == 0 else '❌'} {pkg}")

# Pin numpy LAST — must come after everything else
# so nothing overwrites it
print("  Pinning numpy to 1.26.4...")
subprocess.run(
    [sys.executable, '-m', 'pip', 'install',
     'numpy==1.26.4', '--force-reinstall', '-q'],
    capture_output=True)
print("  ✅ numpy==1.26.4 pinned")

print()
print("⚠️  Restarting runtime to apply numpy fix...")
print("   After restart, run Cell 1 (the actual step code)")
print("   You do NOT need to re-run this cell after restart.")

os.kill(os.getpid(), 9)  # forces immediate runtime restart

Installing Notebook 2 libraries...
  ✅ sentence-transformers
  ✅ faiss-cpu
  ✅ rank_bm25
  ✅ groq
  ✅ rouge-score
  ✅ google-generativeai
  Pinning numpy to 1.26.4...


In [1]:
# ============================================================
# Notebook 2 — Step 5: Meeting-Aware Chunking
# ============================================================
# PURPOSE: Split each meeting's turns into semantically
# coherent chunks (Contribution 1) and build a naive
# 100-word baseline for comparison.
#
# Algorithm (3 passes):
#   Pass 1 — absorb short turns (< 5 words) into neighbors
#   Pass 2 — embed turns, compute windowed cosine similarity
#   Pass 3 — detect boundaries at dynamic threshold
#             (25th percentile of similarity scores per meeting)
#
# Extra signals:
#   Speaker transition after >3s silence → boundary bonus
#   Hard cap: max 12 turns per chunk
#   Hard floor: min 30 words per chunk (merge upward)
#
# Output per meeting:
#   {meeting}_smart_chunks.json  ← your contribution
#   {meeting}_naive_chunks.json  ← baseline for comparison
# ============================================================

import os, json
import numpy as np
from datetime import datetime
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from google.colab import drive

drive.mount('/content/drive')
ROOT = '/content/drive/MyDrive/MeetingMind_v2'
with open(f'{ROOT}/config.json') as f:
    CONFIG = json.load(f)

LOG_PATH = f"{CONFIG['logs_dir']}/run_log.txt"
def log(message, level='INFO'):
    timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    icons = {'INFO': '→', 'SUCCESS': '✅', 'WARNING': '⚠️', 'ERROR': '❌'}
    line = f"[{timestamp}] {icons.get(level,'→')} {message}"
    with open(LOG_PATH, 'a') as f:
        f.write(line + '\n')
    print(line)

def save_checkpoint(step_name, data):
    path = f"{CONFIG['checkpoints_dir']}/{step_name}.json"
    cp = {'step': step_name, 'saved_at': datetime.now().isoformat(),
          'version': CONFIG['version'], 'data': data}
    with open(path, 'w') as f:
        json.dump(cp, f, indent=2)
    log(f"Checkpoint saved: {step_name}", 'SUCCESS')

def load_checkpoint(step_name):
    path = f"{CONFIG['checkpoints_dir']}/{step_name}.json"
    if not os.path.exists(path):
        return None
    with open(path) as f:
        cp = json.load(f)
    log(f"Checkpoint loaded: {step_name}")
    return cp['data']

# ── Load masters ──────────────────────────────────────────────
MASTERS = {}
for mid in CONFIG['ami_meetings']:
    path = f"{CONFIG['transcripts_dir']}/{mid}_master.json"
    with open(path) as f:
        MASTERS[mid] = json.load(f)
print(f"✅ Masters loaded: {list(MASTERS.keys())}")

# ── Load embedding model ──────────────────────────────────────
print()
print("Loading embedding model...")
EMBED_MODEL = SentenceTransformer(CONFIG['embedding_model'])
print(f"✅ Model loaded: {CONFIG['embedding_model']}")
print(f"   Embedding dim: "
      f"{EMBED_MODEL.get_sentence_embedding_dimension()}")

os.makedirs(CONFIG['chunks_dir'], exist_ok=True)

# ============================================================
# PASS 1 — Absorb short turns
# ============================================================
def absorb_short_turns(turns, min_words=5):
    """
    Merge turns with fewer than min_words words into
    their predecessor. Short turns like "Yeah", "Right",
    "Okay" carry no semantic content and create noise.
    Returns cleaned turn list.
    """
    if not turns:
        return turns

    cleaned = [turns[0]]
    absorbed = 0

    for turn in turns[1:]:
        # Skip UNKNOWN turns entirely
        if turn['speaker'] == 'UNKNOWN':
            absorbed += 1
            continue
        # Absorb short turns into predecessor
        if turn['word_count'] < min_words and cleaned:
            cleaned[-1] = {
                **cleaned[-1],
                'text'      : cleaned[-1]['text'] + ' ' +
                              turn['text'],
                'end'       : turn['end'],
                'word_count': cleaned[-1]['word_count'] +
                              turn['word_count'],
                'duration'  : round(turn['end'] -
                              cleaned[-1]['start'], 3),
            }
            absorbed += 1
        else:
            if turn['speaker'] != 'UNKNOWN':
                cleaned.append(turn)

    return cleaned, absorbed

# ============================================================
# PASS 2 — Embed turns + windowed similarity
# ============================================================
def compute_similarities(turns, window=2):
    """
    Embed each turn and compute windowed cosine similarity.
    Window=2 means each turn is compared against the average
    of its two neighbors — smooths conversational noise.
    Returns (embeddings, similarities) arrays.
    """
    texts = [t['text'] for t in turns]

    # Embed all turns at once (batch is faster than one-by-one)
    embeddings = EMBED_MODEL.encode(
        texts,
        batch_size=32,
        show_progress_bar=False,
        convert_to_numpy=True,
    )

    # Windowed similarity
    n    = len(embeddings)
    sims = []

    for i in range(1, n):
        # Window: average embeddings in [i-window, i-1]
        window_start = max(0, i - window)
        window_emb   = embeddings[window_start:i].mean(axis=0,
                        keepdims=True)
        cur_emb      = embeddings[i:i+1]
        sim          = cosine_similarity(window_emb,
                        cur_emb)[0][0]
        sims.append(float(sim))

    return embeddings, sims

# ============================================================
# PASS 3 — Dynamic threshold + boundary detection
# ============================================================
def detect_boundaries(turns, sims, embeddings,
                       max_turns=12, min_words=30):
    """
    Detect chunk boundaries using dynamic threshold.

    Dynamic threshold = 25th percentile of similarity scores.
    This adapts per meeting — jumpy meetings get lower threshold,
    sustained discussions get higher threshold.

    Extra boundary signals:
    - Speaker change after >3s silence → +0.15 bonus to boundary score
    - Hard cap: max_turns per chunk
    - Hard floor: min_words per chunk (merge upward after splitting)
    """
    if not sims:
        return [list(range(len(turns)))]

    # Dynamic threshold
    threshold = float(np.percentile(sims, 25))
    print(f"    Dynamic threshold: {threshold:.3f} "
          f"(25th pct of {len(sims)} similarities)")
    print(f"    Sim range: min={min(sims):.3f} "
          f"max={max(sims):.3f} "
          f"mean={np.mean(sims):.3f}")

    # Compute boundary scores
    # Higher score = more likely to be a boundary
    boundary_scores = []
    for i, sim in enumerate(sims):
        score = 1.0 - sim  # low similarity = high boundary score

        # Speaker transition bonus
        if i + 1 < len(turns):
            cur_turn  = turns[i]
            next_turn = turns[i + 1]
            time_gap  = next_turn['start'] - cur_turn['end']
            spk_change = (cur_turn['speaker'] !=
                         next_turn['speaker'])
            if spk_change and time_gap > 3.0:
                score += 0.15  # speaker change after silence

        boundary_scores.append(score)

    # Cut at threshold (boundary score > 1 - threshold)
    boundary_threshold = 1.0 - threshold
    boundaries = [i + 1 for i, s in enumerate(boundary_scores)
                  if s > boundary_threshold]

    # Build initial chunk groups
    chunks_indices = []
    prev = 0
    for b in boundaries:
        chunks_indices.append(list(range(prev, b)))
        prev = b
    chunks_indices.append(list(range(prev, len(turns))))

    # Apply hard cap: split chunks with too many turns
    capped = []
    for chunk in chunks_indices:
        if len(chunk) > max_turns:
            for i in range(0, len(chunk), max_turns):
                capped.append(chunk[i:i + max_turns])
        else:
            capped.append(chunk)

    # Apply hard floor: merge tiny chunks upward
    merged = []
    for chunk in capped:
        words = sum(turns[i]['word_count'] for i in chunk)
        if words < min_words and merged:
            merged[-1].extend(chunk)
        else:
            merged.append(chunk)

    return merged, threshold

# ============================================================
# Build smart chunks from groups
# ============================================================
def build_smart_chunks(turns, chunk_groups,
                        meeting_id, topic_offset=0):
    """
    Convert turn index groups into chunk dicts with metadata.
    """
    chunks = []
    for topic_idx, group in enumerate(chunk_groups):
        group_turns = [turns[i] for i in group]
        if not group_turns:
            continue

        text     = ' '.join(t['speaker'] + ': ' + t['text']
                           for t in group_turns)
        speakers = list(dict.fromkeys(
            t['speaker'] for t in group_turns
        ))  # ordered, no duplicates

        # Primary speaker = who spoke most words
        spk_words = {}
        for t in group_turns:
            spk_words[t['speaker']] = (
                spk_words.get(t['speaker'], 0) + t['word_count']
            )
        primary = max(spk_words, key=spk_words.get)

        chunks.append({
            'chunk_index'    : len(chunks),
            'meeting_id'     : meeting_id,
            'topic_index'    : topic_offset + topic_idx,
            'text'           : text,
            'speakers'       : speakers,
            'primary_speaker': primary,
            'start_time'     : group_turns[0]['start'],
            'end_time'       : group_turns[-1]['end'],
            'word_count'     : sum(t['word_count']
                                  for t in group_turns),
            'turn_count'     : len(group_turns),
            'turn_indices'   : group,
        })
    return chunks

# ============================================================
# Build naive chunks (baseline)
# ============================================================
def build_naive_chunks(turns, meeting_id,
                        chunk_size=100):
    """
    Naive baseline: flatten all text and split every
    chunk_size words. No speaker awareness, no topic detection.
    This is what standard off-the-shelf RAG would produce.
    """
    # Flatten all non-UNKNOWN turns into word list
    all_words = []
    for turn in turns:
        if turn['speaker'] == 'UNKNOWN':
            continue
        for word in turn['text'].split():
            all_words.append(word)

    # Split into fixed-size chunks
    chunks = []
    for i in range(0, len(all_words), chunk_size):
        word_slice = all_words[i:i + chunk_size]
        chunks.append({
            'chunk_index' : len(chunks),
            'meeting_id'  : meeting_id,
            'text'        : ' '.join(word_slice),
            'word_count'  : len(word_slice),
            # No speaker metadata — that's the point
            'speakers'    : [],
            'primary_speaker': None,
            'start_time'  : None,
            'end_time'    : None,
            'topic_index' : None,
        })
    return chunks

# ============================================================
# Full Step 5 pipeline per meeting
# ============================================================
def chunk_meeting(meeting_id):
    master = MASTERS[meeting_id]
    turns  = master['turns']

    print(f"  Input: {len(turns)} turns, "
          f"{master['total_words']:,} words")

    # ── Pass 1: absorb short turns ────────────────────────
    clean_turns, absorbed = absorb_short_turns(turns)
    print(f"  Pass 1: absorbed {absorbed} short/UNKNOWN turns "
          f"→ {len(clean_turns)} clean turns")

    # ── Pass 2: embed + similarities ─────────────────────
    print(f"  Pass 2: embedding {len(clean_turns)} turns...")
    embeddings, sims = compute_similarities(clean_turns,
                                            window=2)
    print(f"  Pass 2: done — {len(sims)} similarity scores")

    # ── Pass 3: boundary detection ────────────────────────
    print(f"  Pass 3: detecting boundaries...")
    chunk_groups, threshold = detect_boundaries(
        clean_turns, sims, embeddings,
        max_turns=CONFIG['max_turns_per_chunk'],
        min_words=CONFIG['min_words_per_chunk'],
    )
    print(f"  Pass 3: {len(chunk_groups)} chunks detected")

    # ── Build smart chunks ────────────────────────────────
    smart = build_smart_chunks(clean_turns, chunk_groups,
                                meeting_id)

    # ── Build naive chunks ────────────────────────────────
    naive = build_naive_chunks(turns, meeting_id,
                                CONFIG['chunk_size_naive'])

    # ── Stats ─────────────────────────────────────────────
    smart_sizes = [c['word_count'] for c in smart]
    naive_sizes = [c['word_count'] for c in naive]

    print(f"  Smart: {len(smart)} chunks  "
          f"avg={np.mean(smart_sizes):.0f} words  "
          f"min={min(smart_sizes)}  "
          f"max={max(smart_sizes)}")
    print(f"  Naive: {len(naive)} chunks  "
          f"avg={np.mean(naive_sizes):.0f} words  "
          f"min={min(naive_sizes)}  "
          f"max={max(naive_sizes)}")

    # ── Sample smart chunks ───────────────────────────────
    print(f"  Sample smart chunks:")
    for c in smart[1:4]:
        preview = c['text'][:80] + ('...' if
                  len(c['text']) > 80 else '')
        print(f"    [{c['start_time']:.0f}s] "
              f"topic={c['topic_index']}  "
              f"{c['word_count']}w  "
              f"spk={c['primary_speaker']}: "
              f"{preview}")

    # ── Save ──────────────────────────────────────────────
    smart_path = (f"{CONFIG['chunks_dir']}/"
                  f"{meeting_id}_smart_chunks.json")
    naive_path = (f"{CONFIG['chunks_dir']}/"
                  f"{meeting_id}_naive_chunks.json")

    with open(smart_path, 'w') as f:
        json.dump(smart, f, indent=2)
    with open(naive_path, 'w') as f:
        json.dump(naive, f, indent=2)

    return {
        'meeting_id'       : meeting_id,
        'smart_count'      : len(smart),
        'naive_count'      : len(naive),
        'smart_avg_words'  : round(np.mean(smart_sizes), 1),
        'naive_avg_words'  : round(np.mean(naive_sizes), 1),
        'smart_path'       : smart_path,
        'naive_path'       : naive_path,
        'threshold'        : round(threshold, 3),
        'topics'           : len(chunk_groups),
    }

# ============================================================
# Run all 3 meetings
# ============================================================
all_stats = {}

for meeting_id in CONFIG['ami_meetings']:
    print()
    print(f"{'='*55}")
    print(f"  {meeting_id}")
    print(f"{'='*55}")

    cp = load_checkpoint(f'step_5_{meeting_id}')
    if cp:
        print(f"  ✅ Already chunked — "
              f"smart={cp['smart_count']}  "
              f"naive={cp['naive_count']}")
        all_stats[meeting_id] = cp
        continue

    try:
        stats = chunk_meeting(meeting_id)
        save_checkpoint(f'step_5_{meeting_id}', stats)
        all_stats[meeting_id] = stats
        log(f"Chunking done: {meeting_id} — "
            f"smart={stats['smart_count']} "
            f"naive={stats['naive_count']}", 'SUCCESS')
    except Exception as e:
        print(f"  ❌ {e}")
        import traceback; traceback.print_exc()

# ============================================================
# Summary table
# ============================================================
print()
print("=" * 60)
print("  Step 5 — Chunking Results")
print("=" * 60)
print(f"  {'Meeting':<12} {'Smart':>6}  {'AvgW':>6}  "
      f"{'Naive':>6}  {'AvgW':>6}  {'Thresh':>7}")
print(f"  {'-'*12} {'-'*6}  {'-'*6}  {'-'*6}  {'-'*6}  {'-'*7}")

for mid, s in all_stats.items():
    print(f"  {mid:<12} "
          f"{s['smart_count']:>6}  "
          f"{s['smart_avg_words']:>6.0f}  "
          f"{s['naive_count']:>6}  "
          f"{s['naive_avg_words']:>6.0f}  "
          f"{s['threshold']:>7.3f}")

print("=" * 60)

save_checkpoint('step_5', {
    'status'  : 'complete',
    'meetings': all_stats,
})
log("Step 5 complete — smart and naive chunks ready", 'SUCCESS')
print()
print("Next: Step 6 — embeddings + FAISS + BM25 index")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Masters loaded: ['EN2001a', 'EN2001b', 'IS1009a']

Loading embedding model...


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/212 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Model loaded: sentence-transformers/multi-qa-mpnet-base-dot-v1
   Embedding dim: 768

  EN2001a


/tmp/ipykernel_6442/2159213923.py:76: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  f"{EMBED_MODEL.get_sentence_embedding_dimension()}")


[2026-05-10 16:44:02] → Checkpoint loaded: step_5_EN2001a
  ✅ Already chunked — smart=105  naive=119

  EN2001b
[2026-05-10 16:44:03] → Checkpoint loaded: step_5_EN2001b
  ✅ Already chunked — smart=70  naive=74

  IS1009a
[2026-05-10 16:44:04] → Checkpoint loaded: step_5_IS1009a
  ✅ Already chunked — smart=16  naive=15

  Step 5 — Chunking Results
  Meeting       Smart    AvgW   Naive    AvgW   Thresh
  ------------ ------  ------  ------  ------  -------
  EN2001a         105     113     119     100    0.440
  EN2001b          70     105      74      99    0.445
  IS1009a          16      92      15      98    0.418
[2026-05-10 16:44:05] ✅ Checkpoint saved: step_5
[2026-05-10 16:44:05] ✅ Step 5 complete — smart and naive chunks ready

Next: Step 6 — embeddings + FAISS + BM25 index


#Step 6

In [2]:
# ============================================================
# Notebook 2 — Step 6: Embeddings + FAISS + BM25 Index
# ============================================================
# PURPOSE: Build search indices for both smart and naive chunks.
#
# Per meeting, we build:
#   smart_index.faiss     ← FAISS vector index for smart chunks
#   smart_metadata.json   ← parallel metadata list
#   smart_bm25.pkl        ← BM25 keyword index
#   naive_index.faiss     ← FAISS vector index for naive chunks
#   naive_metadata.json   ← parallel metadata list
#   naive_bm25.pkl        ← BM25 keyword index
#
# The parallel metadata list is position-aligned with FAISS:
#   FAISS position 7 ↔ metadata[7]
#   This alignment enables speaker-aware filtering in Step 7.
# ============================================================

import os, json, pickle
import numpy as np
import faiss
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
from datetime import datetime
from google.colab import drive

drive.mount('/content/drive')
ROOT = '/content/drive/MyDrive/MeetingMind_v2'
with open(f'{ROOT}/config.json') as f:
    CONFIG = json.load(f)

LOG_PATH = f"{CONFIG['logs_dir']}/run_log.txt"
def log(message, level='INFO'):
    timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    icons = {'INFO': '→', 'SUCCESS': '✅', 'WARNING': '⚠️', 'ERROR': '❌'}
    line = f"[{timestamp}] {icons.get(level,'→')} {message}"
    with open(LOG_PATH, 'a') as f:
        f.write(line + '\n')
    print(line)

def save_checkpoint(step_name, data):
    path = f"{CONFIG['checkpoints_dir']}/{step_name}.json"
    cp = {'step': step_name, 'saved_at': datetime.now().isoformat(),
          'version': CONFIG['version'], 'data': data}
    with open(path, 'w') as f:
        json.dump(cp, f, indent=2)
    log(f"Checkpoint saved: {step_name}", 'SUCCESS')

def load_checkpoint(step_name):
    path = f"{CONFIG['checkpoints_dir']}/{step_name}.json"
    if not os.path.exists(path):
        return None
    with open(path) as f:
        cp = json.load(f)
    log(f"Checkpoint loaded: {step_name}")
    return cp['data']

# ── Load embedding model ──────────────────────────────────────
print("Loading embedding model...")
EMBED_MODEL = SentenceTransformer(CONFIG['embedding_model'])
DIM = EMBED_MODEL.get_embedding_dimension()
print(f"✅ Model loaded — dim={DIM}")
print()

os.makedirs(CONFIG['indices_dir'], exist_ok=True)

# ============================================================
# Core functions
# ============================================================

MIN_CHUNK_WORDS = 20  # drop chunks below this — too small for RAG

def filter_chunks(chunks):
    """Drop chunks with fewer than MIN_CHUNK_WORDS words."""
    filtered = [c for c in chunks
                if c['word_count'] >= MIN_CHUNK_WORDS]
    dropped  = len(chunks) - len(filtered)
    return filtered, dropped

def embed_chunks(chunks):
    """
    Embed all chunks and return float32 numpy array.
    Shape: (n_chunks, 768)
    FAISS requires float32 specifically.
    """
    texts = [c['text'] for c in chunks]
    embs  = EMBED_MODEL.encode(
        texts,
        batch_size       = 32,
        show_progress_bar= False,
        convert_to_numpy = True,
    )
    return embs.astype(np.float32)

def build_faiss_index(embeddings):
    """
    Build a FAISS flat inner-product index.

    We use IndexFlatIP (inner product) rather than
    IndexFlatL2 (euclidean distance) because our embeddings
    are normalized — inner product equals cosine similarity
    for normalized vectors. Cosine similarity is the right
    metric for semantic text retrieval.
    """
    # Normalize embeddings so IP = cosine similarity
    faiss.normalize_L2(embeddings)
    index = faiss.IndexFlatIP(embeddings.shape[1])
    index.add(embeddings)
    return index

def build_bm25_index(chunks):
    """
    Build BM25 keyword index.
    Tokenize by splitting on whitespace and lowercasing.
    BM25Okapi is the standard Okapi BM25 implementation.
    """
    tokenized = [c['text'].lower().split() for c in chunks]
    return BM25Okapi(tokenized)

def save_index(index, path):
    """Save FAISS index to disk."""
    faiss.write_index(index, path)

def load_index(path):
    """Load FAISS index from disk."""
    return faiss.read_index(path)

def build_metadata(chunks):
    """
    Build parallel metadata list.
    Position i in this list corresponds to position i in FAISS.
    We store everything needed for retrieval and display.
    """
    metadata = []
    for c in chunks:
        metadata.append({
            'chunk_index'    : c['chunk_index'],
            'meeting_id'     : c['meeting_id'],
            'text'           : c['text'],
            'speakers'       : c.get('speakers', []),
            'primary_speaker': c.get('primary_speaker', None),
            'start_time'     : c.get('start_time', None),
            'end_time'       : c.get('end_time', None),
            'topic_index'    : c.get('topic_index', None),
            'word_count'     : c['word_count'],
        })
    return metadata

# ============================================================
# Test query function — spot check index quality
# ============================================================
def test_query(query, faiss_index, bm25_index,
               metadata, label, k=3):
    """
    Run a test query against both FAISS and BM25.
    Prints top-k results so we can visually confirm
    the index is retrieving sensible content.
    """
    print(f"  Query: '{query}'")

    # FAISS search
    q_emb = EMBED_MODEL.encode([query],
                                convert_to_numpy=True
                               ).astype(np.float32)
    faiss.normalize_L2(q_emb)
    scores, indices = faiss_index.search(q_emb, k)

    print(f"  FAISS top-{k}:")
    for rank, (idx, score) in enumerate(
            zip(indices[0], scores[0])):
        if idx < len(metadata):
            m       = metadata[idx]
            preview = m['text'][:60] + '...'
            spk     = m.get('primary_speaker', 'N/A')
            print(f"    {rank+1}. [{score:.3f}] "
                  f"{spk}: {preview}")

    # BM25 search
    tokens   = query.lower().split()
    bm25_scores = bm25_index.get_scores(tokens)
    top_bm25 = np.argsort(bm25_scores)[::-1][:k]

    print(f"  BM25 top-{k}:")
    for rank, idx in enumerate(top_bm25):
        if idx < len(metadata):
            m       = metadata[idx]
            preview = m['text'][:60] + '...'
            print(f"    {rank+1}. [{bm25_scores[idx]:.3f}] "
                  f"{preview}")
    print()

# ============================================================
# Full Step 6 pipeline per meeting
# ============================================================
def index_meeting(meeting_id):
    chunks_dir  = CONFIG['chunks_dir']
    indices_dir = CONFIG['indices_dir']

    # ── Load chunks ───────────────────────────────────────
    smart_path = f"{chunks_dir}/{meeting_id}_smart_chunks.json"
    naive_path = f"{chunks_dir}/{meeting_id}_naive_chunks.json"

    with open(smart_path) as f:
        smart_chunks = json.load(f)
    with open(naive_path) as f:
        naive_chunks = json.load(f)

    print(f"  Loaded: {len(smart_chunks)} smart, "
          f"{len(naive_chunks)} naive chunks")

    # ── Filter tiny chunks ────────────────────────────────
    smart_chunks, smart_dropped = filter_chunks(smart_chunks)
    naive_chunks, naive_dropped = filter_chunks(naive_chunks)
    print(f"  After filter (min {MIN_CHUNK_WORDS}w): "
          f"{len(smart_chunks)} smart "
          f"(dropped {smart_dropped}), "
          f"{len(naive_chunks)} naive "
          f"(dropped {naive_dropped})")

    # ── Embed ─────────────────────────────────────────────
    print(f"  Embedding smart chunks...")
    smart_embs = embed_chunks(smart_chunks)
    print(f"  Embedding naive chunks...")
    naive_embs = embed_chunks(naive_chunks)
    print(f"  Embeddings: smart={smart_embs.shape} "
          f"naive={naive_embs.shape}")

    # ── Build FAISS indices ───────────────────────────────
    smart_faiss = build_faiss_index(smart_embs.copy())
    naive_faiss = build_faiss_index(naive_embs.copy())
    print(f"  FAISS: smart={smart_faiss.ntotal} vectors, "
          f"naive={naive_faiss.ntotal} vectors")

    # ── Build BM25 indices ────────────────────────────────
    smart_bm25 = build_bm25_index(smart_chunks)
    naive_bm25 = build_bm25_index(naive_chunks)
    print(f"  BM25: smart={len(smart_chunks)} docs, "
          f"naive={len(naive_chunks)} docs")

    # ── Build metadata lists ──────────────────────────────
    smart_meta = build_metadata(smart_chunks)
    naive_meta = build_metadata(naive_chunks)

    # ── Save everything ───────────────────────────────────
    prefix = f"{indices_dir}/{meeting_id}"

    faiss.write_index(smart_faiss,
                      f"{prefix}_smart.faiss")
    faiss.write_index(naive_faiss,
                      f"{prefix}_naive.faiss")

    with open(f"{prefix}_smart_metadata.json", 'w') as f:
        json.dump(smart_meta, f, indent=2)
    with open(f"{prefix}_naive_metadata.json", 'w') as f:
        json.dump(naive_meta, f, indent=2)

    with open(f"{prefix}_smart_bm25.pkl", 'wb') as f:
        pickle.dump(smart_bm25, f)
    with open(f"{prefix}_naive_bm25.pkl", 'wb') as f:
        pickle.dump(naive_bm25, f)

    # ── Spot check with test query ────────────────────────
    print(f"  Spot check query:")
    test_query(
        "What was the main topic discussed?",
        smart_faiss, smart_bm25, smart_meta,
        label=f"{meeting_id} smart",
    )

    # ── File sizes ────────────────────────────────────────
    files = {
        'smart.faiss' : f"{prefix}_smart.faiss",
        'naive.faiss' : f"{prefix}_naive.faiss",
        'smart_meta'  : f"{prefix}_smart_metadata.json",
        'naive_meta'  : f"{prefix}_naive_metadata.json",
    }
    sizes = {k: round(os.path.getsize(v)/1024, 1)
             for k, v in files.items()}

    return {
        'meeting_id'         : meeting_id,
        'smart_chunks'       : len(smart_chunks),
        'naive_chunks'       : len(naive_chunks),
        'smart_dropped'      : smart_dropped,
        'naive_dropped'      : naive_dropped,
        'embedding_dim'      : DIM,
        'file_sizes_kb'      : sizes,
    }

# ============================================================
# Run all 3 meetings
# ============================================================
all_stats = {}

for meeting_id in CONFIG['ami_meetings']:
    print()
    print(f"{'='*55}")
    print(f"  {meeting_id}")
    print(f"{'='*55}")

    cp = load_checkpoint(f'step_6_{meeting_id}')
    if cp:
        print(f"  ✅ Already indexed — "
              f"smart={cp['smart_chunks']} "
              f"naive={cp['naive_chunks']}")
        all_stats[meeting_id] = cp
        continue

    try:
        stats = index_meeting(meeting_id)
        save_checkpoint(f'step_6_{meeting_id}', stats)
        all_stats[meeting_id] = stats
        log(f"Indexing done: {meeting_id}", 'SUCCESS')
    except Exception as e:
        print(f"  ❌ {e}")
        import traceback; traceback.print_exc()

# ============================================================
# Summary
# ============================================================
print()
print("=" * 60)
print("  Step 6 — Index Results")
print("=" * 60)
print(f"  {'Meeting':<12} {'Smart':>6}  {'Naive':>6}  "
      f"{'Dropped':>8}")
print(f"  {'-'*12} {'-'*6}  {'-'*6}  {'-'*8}")

for mid, s in all_stats.items():
    print(f"  {mid:<12} "
          f"{s['smart_chunks']:>6}  "
          f"{s['naive_chunks']:>6}  "
          f"{s['smart_dropped']+s['naive_dropped']:>8}")

print("=" * 60)

save_checkpoint('step_6', {
    'status'  : 'complete',
    'meetings': all_stats,
})
log("Step 6 complete — all indices ready", 'SUCCESS')
print()
print("Next: Step 7 — speaker-aware retrieval vs blind retrieval")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loading embedding model...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/multi-qa-mpnet-base-dot-v1
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Model loaded — dim=768


  EN2001a
[2026-05-10 16:44:23] → Checkpoint loaded: step_6_EN2001a
  ✅ Already indexed — smart=104 naive=119

  EN2001b
[2026-05-10 16:44:24] → Checkpoint loaded: step_6_EN2001b
  ✅ Already indexed — smart=69 naive=74

  IS1009a
[2026-05-10 16:44:25] → Checkpoint loaded: step_6_IS1009a
  ✅ Already indexed — smart=15 naive=15

  Step 6 — Index Results
  Meeting       Smart   Naive   Dropped
  ------------ ------  ------  --------
  EN2001a         104     119         1
  EN2001b          69      74         1
  IS1009a          15      15         1
[2026-05-10 16:44:25] ✅ Checkpoint saved: step_6
[2026-05-10 16:44:25] ✅ Step 6 complete — all indices ready

Next: Step 7 — speaker-aware retrieval vs blind retrieval


#Step 7

In [3]:
# ============================================================
# Notebook 2 — Step 7: Retrieval Layer
# ============================================================
# PURPOSE: Build three retrieval modes and validate them.
#
# Mode 1 — blind_retrieve()     : naive chunks, pure vector
# Mode 2 — smart_retrieve()     : smart chunks, speaker-aware
# Mode 3 — hybrid_retrieve()    : smart chunks, FAISS + BM25
#
# All three modes work across all 3 meetings simultaneously.
# The meeting_id parameter scopes retrieval to one meeting,
# or pass None to search across all meetings.
# ============================================================

import os, json, pickle, re
import numpy as np
import faiss
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
from datetime import datetime
from google.colab import drive

drive.mount('/content/drive')
ROOT = '/content/drive/MyDrive/MeetingMind_v2'
with open(f'{ROOT}/config.json') as f:
    CONFIG = json.load(f)

LOG_PATH = f"{CONFIG['logs_dir']}/run_log.txt"
def log(message, level='INFO'):
    timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    icons = {'INFO': '→', 'SUCCESS': '✅', 'WARNING': '⚠️', 'ERROR': '❌'}
    line = f"[{timestamp}] {icons.get(level,'→')} {message}"
    with open(LOG_PATH, 'a') as f:
        f.write(line + '\n')
    print(line)

def save_checkpoint(step_name, data):
    path = f"{CONFIG['checkpoints_dir']}/{step_name}.json"
    cp = {'step': step_name, 'saved_at': datetime.now().isoformat(),
          'version': CONFIG['version'], 'data': data}
    with open(path, 'w') as f:
        json.dump(cp, f, indent=2)
    log(f"Checkpoint saved: {step_name}", 'SUCCESS')

# ── Load embedding model ──────────────────────────────────────
print("Loading embedding model...")
EMBED_MODEL = SentenceTransformer(CONFIG['embedding_model'])
print(f"✅ Model loaded")

# ============================================================
# Load all indices into memory
# ============================================================
print()
print("Loading indices from Drive...")

INDICES = {}   # meeting_id → {smart_faiss, naive_faiss,
               #               smart_bm25, naive_bm25,
               #               smart_meta, naive_meta}

for mid in CONFIG['ami_meetings']:
    prefix = f"{CONFIG['indices_dir']}/{mid}"

    smart_faiss = faiss.read_index(f"{prefix}_smart.faiss")
    naive_faiss = faiss.read_index(f"{prefix}_naive.faiss")

    with open(f"{prefix}_smart_bm25.pkl", 'rb') as f:
        smart_bm25 = pickle.load(f)
    with open(f"{prefix}_naive_bm25.pkl", 'rb') as f:
        naive_bm25 = pickle.load(f)

    with open(f"{prefix}_smart_metadata.json") as f:
        smart_meta = json.load(f)
    with open(f"{prefix}_naive_metadata.json") as f:
        naive_meta = json.load(f)

    INDICES[mid] = {
        'smart_faiss': smart_faiss,
        'naive_faiss': naive_faiss,
        'smart_bm25' : smart_bm25,
        'naive_bm25' : naive_bm25,
        'smart_meta' : smart_meta,
        'naive_meta' : naive_meta,
    }
    print(f"  ✅ {mid}: smart={smart_faiss.ntotal} "
          f"naive={naive_faiss.ntotal} vectors")

print()

# ============================================================
# Speaker detection
# ============================================================

# Maps position words to SPEAKER labels
POSITION_MAP = {
    'first'  : 'SPEAKER_00',
    'second' : 'SPEAKER_01',
    'third'  : 'SPEAKER_02',
    'fourth' : 'SPEAKER_03',
    'fifth'  : 'SPEAKER_04',
    '1st'    : 'SPEAKER_00',
    '2nd'    : 'SPEAKER_01',
    '3rd'    : 'SPEAKER_02',
    '4th'    : 'SPEAKER_03',
}

def detect_speaker(query):
    """
    Detect if query mentions a specific speaker.
    Returns speaker label string or None.

    Handles:
    - "what did SPEAKER_01 say"
    - "what did the second speaker say"
    - "SPEAKER_00's opinion"
    """
    query_lower = query.lower()

    # Direct label match: SPEAKER_XX
    direct = re.search(r'speaker[_\s]?(\d+)', query_lower)
    if direct:
        return f"SPEAKER_{int(direct.group(1)):02d}"

    # Position reference
    for word, label in POSITION_MAP.items():
        if word in query_lower and 'speaker' in query_lower:
            return label

    return None  # no speaker mentioned

# ============================================================
# Core retrieval functions
# ============================================================

def embed_query(query):
    """Embed query and normalize for cosine similarity."""
    emb = EMBED_MODEL.encode(
        [query], convert_to_numpy=True
    ).astype(np.float32)
    faiss.normalize_L2(emb)
    return emb

def get_meetings(meeting_id):
    """Return list of meeting IDs to search."""
    if meeting_id:
        return [meeting_id]
    return CONFIG['ami_meetings']

def blind_retrieve(query, k=5, meeting_id=None):
    """
    Baseline retrieval: naive chunks, pure vector search.
    No speaker awareness. This is standard off-the-shelf RAG.
    """
    q_emb    = embed_query(query)
    meetings = get_meetings(meeting_id)
    results  = []

    for mid in meetings:
        idx    = INDICES[mid]
        scores, positions = idx['naive_faiss'].search(q_emb, k)
        for score, pos in zip(scores[0], positions[0]):
            if pos < len(idx['naive_meta']):
                result = dict(idx['naive_meta'][pos])
                result['score']      = float(score)
                result['meeting_id'] = mid
                result['mode']       = 'blind'
                results.append(result)

    # Sort by score, return top-k
    results.sort(key=lambda x: x['score'], reverse=True)
    return results[:k]

def smart_retrieve(query, k=5, meeting_id=None):
    """
    Contribution 2: speaker-aware retrieval on smart chunks.

    If speaker detected in query:
      → filter metadata to that speaker's chunks
      → search within filtered subset only
    Else:
      → search all chunks normally
    """
    q_emb    = embed_query(query)
    speaker  = detect_speaker(query)
    meetings = get_meetings(meeting_id)
    results  = []

    for mid in meetings:
        idx  = INDICES[mid]
        meta = idx['smart_meta']

        if speaker:
            # Filter to speaker's chunks
            filtered_positions = [
                i for i, m in enumerate(meta)
                if m.get('primary_speaker') == speaker
                or speaker in m.get('speakers', [])
            ]

            if not filtered_positions:
                # Speaker not found in this meeting — skip
                continue

            # Extract those vectors from FAISS
            filtered_vectors = np.zeros(
                (len(filtered_positions),
                 q_emb.shape[1]),
                dtype=np.float32
            )
            for i, pos in enumerate(filtered_positions):
                filtered_vectors[i] = \
                    idx['smart_faiss'].reconstruct(pos)

            # Build temporary mini-index for filtered subset
            mini_idx = faiss.IndexFlatIP(q_emb.shape[1])
            faiss.normalize_L2(filtered_vectors)
            mini_idx.add(filtered_vectors)

            n_search = min(k, len(filtered_positions))
            scores, mini_pos = mini_idx.search(
                q_emb, n_search)

            for score, mpos in zip(scores[0], mini_pos[0]):
                if mpos >= 0 and mpos < len(filtered_positions):
                    orig_pos = filtered_positions[mpos]
                    result   = dict(meta[orig_pos])
                    result['score']         = float(score)
                    result['meeting_id']    = mid
                    result['mode']          = 'smart_filtered'
                    result['filter_speaker']= speaker
                    results.append(result)
        else:
            # No speaker filter — search all chunks
            scores, positions = idx['smart_faiss'].search(
                q_emb, k)
            for score, pos in zip(scores[0], positions[0]):
                if pos < len(meta):
                    result = dict(meta[pos])
                    result['score']      = float(score)
                    result['meeting_id'] = mid
                    result['mode']       = 'smart_all'
                    results.append(result)

    results.sort(key=lambda x: x['score'], reverse=True)
    return results[:k]

def hybrid_retrieve(query, k=5, meeting_id=None,
                    alpha=None):
    """
    Contribution 3: hybrid FAISS + BM25 retrieval.

    Score formula:
      final = alpha × normalized_vector_score
            + (1-alpha) × normalized_bm25_score

    Normalization: min-max so both scores are in [0,1]
    before combining. Without normalization the scales
    are incompatible (FAISS ~0.5, BM25 ~0-10).

    Speaker filtering applied same as smart_retrieve.
    """
    if alpha is None:
        alpha = CONFIG['hybrid_alpha']  # 0.5

    q_emb    = embed_query(query)
    speaker  = detect_speaker(query)
    tokens   = query.lower().split()
    meetings = get_meetings(meeting_id)
    results  = []

    for mid in meetings:
        idx  = INDICES[mid]
        meta = idx['smart_meta']
        n    = len(meta)

        # ── Vector scores ─────────────────────────────────
        scores_v, positions_v = idx['smart_faiss'].search(
            q_emb, n)
        vector_scores = np.zeros(n)
        for score, pos in zip(scores_v[0], positions_v[0]):
            if 0 <= pos < n:
                vector_scores[pos] = float(score)

        # ── BM25 scores ───────────────────────────────────
        bm25_scores = np.array(
            idx['smart_bm25'].get_scores(tokens))

        # ── Normalize both to [0, 1] ──────────────────────
        def normalize(arr):
            mn, mx = arr.min(), arr.max()
            if mx - mn < 1e-10:
                return np.zeros_like(arr)
            return (arr - mn) / (mx - mn)

        v_norm   = normalize(vector_scores)
        bm25_norm= normalize(bm25_scores)

        # ── Combine ───────────────────────────────────────
        combined = alpha * v_norm + (1 - alpha) * bm25_norm

        # ── Speaker filter ────────────────────────────────
        if speaker:
            for i, m in enumerate(meta):
                if (m.get('primary_speaker') != speaker and
                        speaker not in m.get('speakers', [])):
                    combined[i] = 0.0  # zero out other speakers

        # ── Get top-k ─────────────────────────────────────
        top_positions = np.argsort(combined)[::-1][:k]
        for pos in top_positions:
            if combined[pos] > 0:
                result = dict(meta[pos])
                result['score']        = float(combined[pos])
                result['vector_score'] = float(v_norm[pos])
                result['bm25_score']   = float(bm25_norm[pos])
                result['meeting_id']   = mid
                result['mode']         = 'hybrid'
                results.append(result)

    results.sort(key=lambda x: x['score'], reverse=True)
    return results[:k]

# ============================================================
# Validation — run test queries
# ============================================================
print("=" * 60)
print("  Step 7 — Retrieval Validation")
print("=" * 60)

def print_results(results, label):
    print(f"\n  [{label}]")
    for i, r in enumerate(results[:3]):
        spk     = r.get('primary_speaker', 'N/A')
        score   = r['score']
        preview = r['text'][:70] + '...'
        mid     = r.get('meeting_id', '?')
        print(f"    {i+1}. [{score:.3f}] {mid} {spk}: {preview}")

# ── Test 1: General question ──────────────────────────────────
print("\n── Test 1: General question (all meetings) ──────────────")
q1 = "What decisions were made about the project?"
print(f"  Query: '{q1}'")
print_results(blind_retrieve(q1,  k=3), "Blind (naive)")
print_results(smart_retrieve(q1,  k=3), "Smart (speaker-aware)")
print_results(hybrid_retrieve(q1, k=3), "Hybrid (FAISS+BM25)")

# ── Test 2: Speaker-specific question ─────────────────────────
print("\n── Test 2: Speaker-specific (EN2001a) ───────────────────")
q2 = "What did SPEAKER_01 say about the interface design?"
print(f"  Query: '{q2}'")
detected = detect_speaker(q2)
print(f"  Detected speaker: {detected}")
print_results(blind_retrieve(q2,  k=3, meeting_id='EN2001a'),
              "Blind")
print_results(smart_retrieve(q2,  k=3, meeting_id='EN2001a'),
              "Smart (filtered to SPEAKER_01)")
print_results(hybrid_retrieve(q2, k=3, meeting_id='EN2001a'),
              "Hybrid (filtered to SPEAKER_01)")

# ── Test 3: Position reference ────────────────────────────────
print("\n── Test 3: Position reference ───────────────────────────")
q3 = "What did the second speaker say?"
print(f"  Query: '{q3}'")
detected3 = detect_speaker(q3)
print(f"  Detected speaker: {detected3}")
print_results(smart_retrieve(q3, k=3, meeting_id='EN2001b'),
              "Smart (filtered to second speaker)")

# ── Test 4: Factual keyword question ─────────────────────────
print("\n── Test 4: Keyword question (hybrid advantage) ──────────")
q4 = "What was discussed about the remote control?"
print(f"  Query: '{q4}'")
r_smart  = smart_retrieve(q4,  k=3, meeting_id='EN2001a')
r_hybrid = hybrid_retrieve(q4, k=3, meeting_id='EN2001a')
print_results(r_smart,  "Smart (pure vector)")
print_results(r_hybrid, "Hybrid (vector + keyword)")

print()
print("=" * 60)

# ── Save checkpoint ───────────────────────────────────────────
save_checkpoint('step_7', {
    'status'         : 'complete',
    'retrieval_modes': ['blind', 'smart', 'hybrid'],
    'speaker_detection': 'regex + position map',
    'hybrid_alpha'   : CONFIG['hybrid_alpha'],
    'meetings_indexed': CONFIG['ami_meetings'],
})
log("Step 7 complete — retrieval layer ready", 'SUCCESS')
print()
print("Next: Step 8 — LLM enhancements")
print("  Query expansion, re-ranking, speaker profiles,")
print("  chunk summarization, conversational memory")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loading embedding model...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/multi-qa-mpnet-base-dot-v1
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Model loaded

Loading indices from Drive...
  ✅ EN2001a: smart=104 naive=119 vectors
  ✅ EN2001b: smart=69 naive=74 vectors
  ✅ IS1009a: smart=15 naive=15 vectors

  Step 7 — Retrieval Validation

── Test 1: General question (all meetings) ──────────────
  Query: 'What decisions were made about the project?'

  [Blind (naive)]
    1. [0.533] EN2001a None: Didn't we sort of agree that it would be useful to have a demonstrator...
    2. [0.520] IS1009a None: design this thing. Selling price is supposed to be 25 euro. Profit aim...
    3. [0.487] EN2001a None: moment it's also an implementation level. With the data structures, I'...

  [Smart (speaker-aware)]
    1. [0.518] IS1009a SPEAKER_02: SPEAKER_02: Here's a project finance. Which of course we all have to t...
    2. [0.478] EN2001a SPEAKER_01: SPEAKER_00: I think it's because we had to specify it ourselves. It's ...
    3. [0.446] EN2001b SPEAKER_03: SPEAKER_01: So how about the prototype if we want to show him somethin...

  [Hy

In [4]:
# ============================================================
# Notebook 2 — Step 8: LLM Enhancements
# ============================================================
# PURPOSE: Add 6 LLM-powered enhancements on top of retrieval.
#
# Enhancement 1 — LLM caller with 4-level fallback chain
# Enhancement 2 — Chunk summarization
# Enhancement 3 — Speaker profile generation
# Enhancement 4 — Query expansion (HyDE)
# Enhancement 5 — LLM re-ranking
# Enhancement 6 — Conversational memory
#
# Fallback chain:
#   Groq llama-3.3-70b → Gemini 2.0 Flash →
#   Groq account 2     → Groq llama-8b
# ============================================================

import os, json, time, re
import numpy as np
import faiss, pickle
from datetime import datetime
from google.colab import drive, userdata
from sentence_transformers import SentenceTransformer
import google.generativeai as genai
from groq import Groq

drive.mount('/content/drive')
ROOT = '/content/drive/MyDrive/MeetingMind_v2'
with open(f'{ROOT}/config.json') as f:
    CONFIG = json.load(f)

LOG_PATH = f"{CONFIG['logs_dir']}/run_log.txt"
def log(message, level='INFO'):
    timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    icons = {'INFO': '→', 'SUCCESS': '✅', 'WARNING': '⚠️', 'ERROR': '❌'}
    line = f"[{timestamp}] {icons.get(level,'→')} {message}"
    with open(LOG_PATH, 'a') as f:
        f.write(line + '\n')
    print(line)

def save_checkpoint(step_name, data):
    path = f"{CONFIG['checkpoints_dir']}/{step_name}.json"
    cp = {'step': step_name, 'saved_at': datetime.now().isoformat(),
          'version': CONFIG['version'], 'data': data}
    with open(path, 'w') as f:
        json.dump(cp, f, indent=2)
    log(f"Checkpoint saved: {step_name}", 'SUCCESS')

# ── Load API keys ─────────────────────────────────────────────
GROQ_KEY_1   = userdata.get('GROQ_API_KEY')
GROQ_KEY_2   = userdata.get('GROQ_API_KEY_2')
GEMINI_KEY   = userdata.get('GEMINI_API_KEY')

print("── API Keys ─────────────────────────────────────────────")
print(f"  Groq primary  : {'✅' if GROQ_KEY_1 else '❌'}")
print(f"  Groq fallback : {'✅' if GROQ_KEY_2 else '❌'}")
print(f"  Gemini        : {'✅' if GEMINI_KEY else '❌'}")

# Configure Gemini
if GEMINI_KEY:
    genai.configure(api_key=GEMINI_KEY)

# ============================================================
# ENHANCEMENT 1 — LLM caller with 4-level fallback chain
# ============================================================

def call_llm(prompt, system="You are a helpful assistant.",
             max_tokens=1024, temperature=0.3):
    """
    Call LLM with automatic 4-level fallback chain.

    Level 1: Groq llama-3.3-70b (best quality)
    Level 2: Gemini 2.0 Flash   (comparable quality, 10x quota)
    Level 3: Groq account 2     (same quality, fresh quota)
    Level 4: Groq llama-8b      (always available, lower quality)

    Returns: (response_text, model_used)
    """

    # ── Level 1: Groq primary ────────────────────────────
    if GROQ_KEY_1:
        try:
            client = Groq(api_key=GROQ_KEY_1)
            resp   = client.chat.completions.create(
                model      = CONFIG['groq_model'],
                messages   = [
                    {"role": "system", "content": system},
                    {"role": "user",   "content": prompt},
                ],
                max_tokens  = max_tokens,
                temperature = temperature,
            )
            return resp.choices[0].message.content, \
                   CONFIG['groq_model']
        except Exception as e:
            if 'rate' in str(e).lower() or '429' in str(e):
                log(f"Groq primary rate limited — trying Gemini",
                    'WARNING')
            else:
                log(f"Groq primary error: {str(e)[:60]}",
                    'WARNING')

    # ── Level 2: Gemini 2.0 Flash ────────────────────────
    if GEMINI_KEY:
        try:
            model = genai.GenerativeModel(
                'gemini-2.0-flash',
                system_instruction=system,
            )
            resp = model.generate_content(
                prompt,
                generation_config=genai.types.GenerationConfig(
                    max_output_tokens = max_tokens,
                    temperature       = temperature,
                )
            )
            return resp.text, 'gemini-2.0-flash'
        except Exception as e:
            log(f"Gemini error: {str(e)[:60]}", 'WARNING')

    # ── Level 3: Groq account 2 ──────────────────────────
    if GROQ_KEY_2:
        try:
            client = Groq(api_key=GROQ_KEY_2)
            resp   = client.chat.completions.create(
                model      = CONFIG['groq_model'],
                messages   = [
                    {"role": "system", "content": system},
                    {"role": "user",   "content": prompt},
                ],
                max_tokens  = max_tokens,
                temperature = temperature,
            )
            return resp.choices[0].message.content, \
                   f"{CONFIG['groq_model']} (account 2)"
        except Exception as e:
            log(f"Groq account 2 error: {str(e)[:60]}",
                'WARNING')

    # ── Level 4: Groq llama-8b (last resort) ─────────────
    if GROQ_KEY_1:
        try:
            client = Groq(api_key=GROQ_KEY_1)
            resp   = client.chat.completions.create(
                model      = CONFIG['groq_model_fallback'],
                messages   = [
                    {"role": "system", "content": system},
                    {"role": "user",   "content": prompt},
                ],
                max_tokens  = max_tokens,
                temperature = temperature,
            )
            return resp.choices[0].message.content, \
                   CONFIG['groq_model_fallback']
        except Exception as e:
            log(f"Groq fallback error: {str(e)[:60]}", 'ERROR')

    raise RuntimeError("All LLM providers failed")

# ── Quick test of fallback chain ──────────────────────────────
print()
print("Testing LLM fallback chain...")
test_resp, model_used = call_llm(
    "Say 'LLM working' and nothing else.",
    max_tokens=10
)
print(f"  ✅ Response: '{test_resp.strip()}'")
print(f"  ✅ Model used: {model_used}")

# ============================================================
# ENHANCEMENT 2 — Chunk summarization
# ============================================================

def summarize_chunk(chunk_text, meeting_id):
    """
    Generate a one-sentence summary of a chunk.
    Used for retrieval — cleaner signal than raw text.
    """
    prompt = f"""You are summarizing a segment from a meeting transcript.
Write ONE sentence (max 30 words) that captures the main point.
Be specific — include key topics, decisions, or questions raised.
Do not start with 'The speakers' or 'In this segment'.

Transcript segment:
{chunk_text[:800]}

One-sentence summary:"""

    summary, _ = call_llm(prompt, max_tokens=60,
                           temperature=0.1)
    return summary.strip()

def build_chunk_summaries(meeting_id, force=False):
    """
    Generate summaries for all smart chunks of a meeting.
    Saves to Drive. Loads from cache if already done.
    """
    cache_path = (f"{CONFIG['chunks_dir']}/"
                  f"{meeting_id}_chunk_summaries.json")

    if os.path.exists(cache_path) and not force:
        with open(cache_path) as f:
            summaries = json.load(f)
        print(f"  ✅ Loaded {len(summaries)} summaries from cache")
        return summaries

    # Load smart chunks
    chunks_path = (f"{CONFIG['chunks_dir']}/"
                   f"{meeting_id}_smart_chunks.json")
    with open(chunks_path) as f:
        chunks = [c for c in json.load(f)
                  if c['word_count'] >= 20]

    print(f"  Summarizing {len(chunks)} chunks...")
    summaries = {}

    for i, chunk in enumerate(chunks):
        try:
            summary = summarize_chunk(
                chunk['text'], meeting_id)
            summaries[chunk['chunk_index']] = summary
            if (i + 1) % 10 == 0:
                print(f"    {i+1}/{len(chunks)} done...")
            time.sleep(0.5)  # rate limit buffer
        except Exception as e:
            summaries[chunk['chunk_index']] = chunk['text'][:100]
            log(f"Summary failed chunk {i}: {e}", 'WARNING')

    with open(cache_path, 'w') as f:
        json.dump(summaries, f, indent=2)

    print(f"  ✅ {len(summaries)} summaries saved")
    return summaries

# ============================================================
# ENHANCEMENT 3 — Speaker profile generation
# ============================================================

def generate_speaker_profile(speaker_id, meeting_id,
                              master, force=False):
    """
    Generate a profile for one speaker from their turns.
    Captures: role, communication style, main topics,
    key positions taken in the meeting.
    """
    cache_path = (f"{CONFIG['eval_dir']}/"
                  f"{meeting_id}_{speaker_id}_profile.json")

    if os.path.exists(cache_path) and not force:
        with open(cache_path) as f:
            return json.load(f)

    # Collect speaker's turns
    speaker_turns = [
        t for t in master['turns']
        if t['speaker'] == speaker_id
        and t['word_count'] >= 5
    ]

    if not speaker_turns:
        return {'speaker': speaker_id,
                'profile': 'No turns found.'}

    # Sample up to 20 turns for the prompt
    sample = speaker_turns[:20]
    turns_text = '\n'.join(
        f"- {t['text'][:150]}" for t in sample
    )

    total_words = sum(t['word_count'] for t in speaker_turns)
    total_turns = len(speaker_turns)

    prompt = f"""Analyze this speaker from a meeting transcript.
Speaker ID: {speaker_id}
Meeting: {meeting_id}
Statistics: {total_turns} turns, {total_words} words spoken

Sample of their statements:
{turns_text}

Write a brief profile (3-4 sentences) covering:
1. Their likely role or expertise based on what they discuss
2. Their communication style (concise/verbose, questioning/assertive)
3. The main topics they focused on
4. Any key positions or opinions they expressed

Profile:"""

    profile_text, model = call_llm(
        prompt, max_tokens=200, temperature=0.3)

    profile = {
        'speaker'     : speaker_id,
        'meeting_id'  : meeting_id,
        'total_turns' : total_turns,
        'total_words' : total_words,
        'profile'     : profile_text.strip(),
        'model_used'  : model,
    }

    with open(cache_path, 'w') as f:
        json.dump(profile, f, indent=2)

    return profile

def build_all_profiles(meeting_id, master):
    """Build profiles for all speakers in a meeting."""
    speakers = [s for s in master['speakers']
                if s != 'UNKNOWN']
    profiles = {}
    print(f"  Building profiles for {len(speakers)} speakers...")
    for spk in speakers:
        profile = generate_speaker_profile(
            spk, meeting_id, master)
        profiles[spk] = profile
        print(f"    ✅ {spk}: "
              f"{profile['profile'][:80]}...")
        time.sleep(0.5)
    return profiles

# ============================================================
# ENHANCEMENT 4 — Query expansion (HyDE)
# ============================================================

def expand_query(query, n=3):
    """
    Generate n alternative phrasings of the query.
    Each variation captures a different angle of the
    same question — wider retrieval net.

    Returns list of [original] + [n variations]
    """
    prompt = f"""Generate {n} different ways to ask the same question.
Each variation should capture a different angle or use different keywords.
Return ONLY the {n} questions, one per line, no numbering.

Original question: {query}

{n} variations:"""

    response, _ = call_llm(prompt, max_tokens=150,
                            temperature=0.7)

    variations = [line.strip() for line in
                  response.strip().split('\n')
                  if line.strip() and len(line.strip()) > 10]

    # Always include original + up to n variations
    all_queries = [query] + variations[:n]
    return all_queries

# ============================================================
# ENHANCEMENT 5 — LLM re-ranking
# ============================================================

def rerank_chunks(query, chunks, top_k=5):
    """
    Send retrieved chunks to LLM for re-ranking.
    LLM scores each chunk 1-10 for relevance to query.
    Returns top_k chunks sorted by LLM score.

    This is the cross-encoder re-ranking step —
    more expensive but more precise than vector similarity.
    """
    if not chunks:
        return chunks

    # Build prompt with numbered chunks
    chunks_text = ""
    for i, chunk in enumerate(chunks):
        preview = chunk['text'][:200]
        chunks_text += f"\n[{i+1}] {preview}\n"

    prompt = f"""Rate each excerpt for relevance to the question.
Score 1-10 (10=most relevant). Return ONLY scores as:
1:score, 2:score, 3:score, etc.

Question: {query}

Excerpts:{chunks_text}

Scores:"""

    try:
        response, _ = call_llm(prompt, max_tokens=100,
                                temperature=0.1)

        # Parse scores
        scores = {}
        for match in re.finditer(r'(\d+)\s*:\s*(\d+(?:\.\d+)?)',
                                  response):
            idx   = int(match.group(1)) - 1
            score = float(match.group(2))
            if 0 <= idx < len(chunks):
                scores[idx] = score

        # Sort by LLM score
        ranked = sorted(
            range(len(chunks)),
            key=lambda i: scores.get(i, 0),
            reverse=True
        )
        reranked = [chunks[i] for i in ranked[:top_k]]

        # Add rerank score to metadata
        for chunk, idx in zip(reranked, ranked[:top_k]):
            chunk['rerank_score'] = scores.get(idx, 0)

        return reranked

    except Exception as e:
        log(f"Re-ranking failed: {e} — returning original order",
            'WARNING')
        return chunks[:top_k]

# ============================================================
# ENHANCEMENT 6 — Conversational memory
# ============================================================

CONVERSATION_HISTORY = []
MAX_HISTORY          = 3  # keep last 3 exchanges

def add_to_history(question, answer):
    """Add a Q&A exchange to conversation history."""
    CONVERSATION_HISTORY.append({
        'question': question,
        'answer'  : answer[:300],  # truncate for token budget
    })
    # Keep only last MAX_HISTORY exchanges
    if len(CONVERSATION_HISTORY) > MAX_HISTORY:
        CONVERSATION_HISTORY.pop(0)

def get_history_context():
    """Format conversation history for LLM prompt."""
    if not CONVERSATION_HISTORY:
        return ""
    history_text = "\n".join(
        f"Q: {h['question']}\nA: {h['answer']}"
        for h in CONVERSATION_HISTORY
    )
    return f"\nPrevious conversation:\n{history_text}\n"

def clear_history():
    """Clear conversation history (new topic)."""
    CONVERSATION_HISTORY.clear()
    print("Conversation history cleared.")

# ============================================================
# Load masters for profile generation
# ============================================================
MASTERS = {}
for mid in CONFIG['ami_meetings']:
    path = (f"{CONFIG['transcripts_dir']}/"
            f"{mid}_master.json")
    with open(path) as f:
        MASTERS[mid] = json.load(f)

# ============================================================
# Run all enhancements
# ============================================================
print()
print("=" * 60)
print("  Building LLM Enhancements")
print("=" * 60)

ALL_SUMMARIES = {}
ALL_PROFILES  = {}

for meeting_id in CONFIG['ami_meetings']:
    print()
    print(f"── {meeting_id} ──────────────────────────────────────────")

    # Enhancement 2: chunk summaries
    print(f"  Enhancement 2: chunk summaries")
    summaries = build_chunk_summaries(meeting_id)
    ALL_SUMMARIES[meeting_id] = summaries

    # Enhancement 3: speaker profiles
    print(f"  Enhancement 3: speaker profiles")
    profiles = build_all_profiles(meeting_id,
                                   MASTERS[meeting_id])
    ALL_PROFILES[meeting_id] = profiles

# ── Test query expansion ──────────────────────────────────────
print()
print("── Testing Enhancement 4: Query Expansion ───────────────")
test_q = "What was decided about the project timeline?"
expanded = expand_query(test_q, n=3)
print(f"  Original: {test_q}")
for i, q in enumerate(expanded[1:], 1):
    print(f"  Variation {i}: {q}")

# ── Test re-ranking ───────────────────────────────────────────
print()
print("── Testing Enhancement 5: Re-ranking ────────────────────")

# Load retrieval functions (inline for self-contained cell)
EMBED_MODEL = SentenceTransformer(CONFIG['embedding_model'])
INDICES     = {}
for mid in CONFIG['ami_meetings']:
    prefix = f"{CONFIG['indices_dir']}/{mid}"
    with open(f"{prefix}_smart_metadata.json") as f:
        smart_meta = json.load(f)
    INDICES[mid] = {
        'smart_faiss': faiss.read_index(
            f"{prefix}_smart.faiss"),
        'smart_meta' : smart_meta,
    }

def quick_retrieve(query, meeting_id, k=10):
    """Quick vector retrieval for re-ranking test."""
    emb = EMBED_MODEL.encode(
        [query], convert_to_numpy=True
    ).astype(np.float32)
    faiss.normalize_L2(emb)
    idx    = INDICES[meeting_id]
    scores, positions = idx['smart_faiss'].search(emb, k)
    results = []
    for score, pos in zip(scores[0], positions[0]):
        if pos < len(idx['smart_meta']):
            r = dict(idx['smart_meta'][pos])
            r['score'] = float(score)
            results.append(r)
    return results

test_query_rerank = "What technical challenges were discussed?"
raw_results = quick_retrieve(test_query_rerank, 'EN2001a', k=5)
reranked    = rerank_chunks(test_query_rerank, raw_results)

print(f"  Query: '{test_query_rerank}'")
print(f"  Before re-ranking (top 3):")
for i, r in enumerate(raw_results[:3]):
    print(f"    {i+1}. [{r['score']:.3f}] "
          f"{r['text'][:60]}...")
print(f"  After re-ranking (top 3):")
for i, r in enumerate(reranked[:3]):
    print(f"    {i+1}. [rerank={r.get('rerank_score',0):.0f}] "
          f"{r['text'][:60]}...")

# ── Test conversational memory ────────────────────────────────
print()
print("── Testing Enhancement 6: Conversational Memory ─────────")
add_to_history(
    "What was the main topic?",
    "The team discussed remote control interface design."
)
history = get_history_context()
print(f"  History context preview:")
print(f"  {history.strip()[:150]}...")
clear_history()

# ── Save checkpoint ───────────────────────────────────────────
print()
save_checkpoint('step_8', {
    'status'        : 'complete',
    'enhancements'  : [
        'llm_fallback_chain',
        'chunk_summarization',
        'speaker_profiles',
        'query_expansion',
        'llm_reranking',
        'conversational_memory',
    ],
    'models_available': {
        'primary' : CONFIG['groq_model'],
        'gemini'  : 'gemini-2.0-flash',
        'fallback2': CONFIG['groq_model'],
        'last_resort': CONFIG['groq_model_fallback'],
    },
    'summaries_built' : {mid: len(s)
                         for mid, s in ALL_SUMMARIES.items()},
    'profiles_built'  : {mid: len(p)
                         for mid, p in ALL_PROFILES.items()},
})
log("Step 8 complete — all LLM enhancements ready", 'SUCCESS')
print()
print("Next: Step 9 — assemble full answer_question() pipeline")

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
── API Keys ─────────────────────────────────────────────
  Groq primary  : ✅
  Groq fallback : ✅
  Gemini        : ✅

Testing LLM fallback chain...
  ✅ Response: 'LLM working'
  ✅ Model used: llama-3.3-70b-versatile

  Building LLM Enhancements

── EN2001a ──────────────────────────────────────────
  Enhancement 2: chunk summaries
  ✅ Loaded 104 summaries from cache
  Enhancement 3: speaker profiles
  Building profiles for 5 speakers...
    ✅ SPEAKER_00: Based on the meeting transcript, SPEAKER_00 appears to be a technical expert, li...
    ✅ SPEAKER_01: SPEAKER_01 is likely a technical expert, possibly a software developer or engine...
    ✅ SPEAKER_02: SPEAKER_02 appears to be a technical expert, likely a software developer or engi...
    ✅ SPEAKER_03: SPEAKER_03 appears to be a technical expert, likely with a background in data an...
    ✅ SPEAKER_04: SPE

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/multi-qa-mpnet-base-dot-v1
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Query: 'What technical challenges were discussed?'
  Before re-ranking (top 3):
    1. [0.518] SPEAKER_01: When do we have to meet again then? SPEAKER_01: ...
    2. [0.507] SPEAKER_00: It's a very boring task. It's also complicated. ...
    3. [0.502] SPEAKER_02: Is there anything else we should discuss? SPEAKE...
  After re-ranking (top 3):
    1. [rerank=9] SPEAKER_01: But I'm still confused because I thought like th...
    2. [rerank=8] SPEAKER_02: Does anyone want to see Steve's feedback from th...
    3. [rerank=6] SPEAKER_00: It's a very boring task. It's also complicated. ...

── Testing Enhancement 6: Conversational Memory ─────────
  History context preview:
  Previous conversation:
Q: What was the main topic?
A: The team discussed remote control interface design....
Conversation history cleared.

[2026-05-10 16:45:43] ✅ Checkpoint saved: step_8
[2026-05-10 16:45:43] ✅ Step 8 complete — all LLM enhancements ready

Next: Step 9 — assemble full answer_question() pipeline


In [5]:
# ============================================================
# Notebook 2 — Step 9: Full Pipeline Assembly
# ============================================================
# PURPOSE: Assemble all components into the final
# answer_question() and summarize() functions.
#
# answer_question() pipeline:
#   1. Detect speaker in query
#   2. Expand query → 4 variations (Enhancement 4)
#   3. Hybrid retrieve for all variations (Contribution 3)
#   4. Merge + deduplicate results
#   5. LLM re-rank top-10 (Enhancement 5)
#   6. Inject speaker profile if relevant (Enhancement 3)
#   7. Inject conversation history (Enhancement 6)
#   8. Generate answer with LLM (fallback chain)
#   9. Store in history
#
# Three comparison modes for evaluation:
#   mode='naive'    — blind retrieval, no enhancements
#   mode='smart'    — speaker-aware, no LLM enhancements
#   mode='full'     — everything enabled
# ============================================================

import os, json, time, re, pickle
import numpy as np
import faiss
from datetime import datetime
from google.colab import drive, userdata
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi
from groq import Groq
from google import genai as google_genai

drive.mount('/content/drive')
ROOT = '/content/drive/MyDrive/MeetingMind_v2'
with open(f'{ROOT}/config.json') as f:
    CONFIG = json.load(f)

LOG_PATH = f"{CONFIG['logs_dir']}/run_log.txt"
def log(message, level='INFO'):
    timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    icons = {'INFO': '→', 'SUCCESS': '✅', 'WARNING': '⚠️', 'ERROR': '❌'}
    line = f"[{timestamp}] {icons.get(level,'→')} {message}"
    with open(LOG_PATH, 'a') as f:
        f.write(line + '\n')
    print(line)

def save_checkpoint(step_name, data):
    path = f"{CONFIG['checkpoints_dir']}/{step_name}.json"
    cp = {'step': step_name, 'saved_at': datetime.now().isoformat(),
          'version': CONFIG['version'], 'data': data}
    with open(path, 'w') as f:
        json.dump(cp, f, indent=2)
    log(f"Checkpoint saved: {step_name}", 'SUCCESS')

# ── Load API keys ─────────────────────────────────────────────
GROQ_KEY_1 = userdata.get('GROQ_API_KEY')
GROQ_KEY_2 = userdata.get('GROQ_API_KEY_2')
GEMINI_KEY = userdata.get('GEMINI_API_KEY')

# Configure new Gemini client
gemini_client = None
if GEMINI_KEY:
    gemini_client = google_genai.Client(api_key=GEMINI_KEY)

# ── Load embedding model ──────────────────────────────────────
print("Loading embedding model...")
EMBED_MODEL = SentenceTransformer(CONFIG['embedding_model'])
print("✅ Model loaded")

# ── Load all indices ──────────────────────────────────────────
print("Loading indices...")
INDICES = {}
for mid in CONFIG['ami_meetings']:
    prefix = f"{CONFIG['indices_dir']}/{mid}"
    with open(f"{prefix}_smart_metadata.json") as f:
        smart_meta = json.load(f)
    with open(f"{prefix}_naive_metadata.json") as f:
        naive_meta = json.load(f)
    with open(f"{prefix}_smart_bm25.pkl", 'rb') as f:
        smart_bm25 = pickle.load(f)
    with open(f"{prefix}_naive_bm25.pkl", 'rb') as f:
        naive_bm25 = pickle.load(f)
    INDICES[mid] = {
        'smart_faiss': faiss.read_index(
            f"{prefix}_smart.faiss"),
        'naive_faiss': faiss.read_index(
            f"{prefix}_naive.faiss"),
        'smart_bm25' : smart_bm25,
        'naive_bm25' : naive_bm25,
        'smart_meta' : smart_meta,
        'naive_meta' : naive_meta,
    }
    print(f"  ✅ {mid}")

# ── Load masters + profiles + summaries ──────────────────────
print("Loading masters and profiles...")
MASTERS   = {}
PROFILES  = {}
SUMMARIES = {}

for mid in CONFIG['ami_meetings']:
    with open(f"{CONFIG['transcripts_dir']}/"
              f"{mid}_master.json") as f:
        MASTERS[mid] = json.load(f)

    # Load speaker profiles
    PROFILES[mid] = {}
    for spk in MASTERS[mid]['speakers']:
        if spk == 'UNKNOWN':
            continue
        profile_path = (f"{CONFIG['eval_dir']}/"
                       f"{mid}_{spk}_profile.json")
        if os.path.exists(profile_path):
            with open(profile_path) as f:
                PROFILES[mid][spk] = json.load(f)

    # Load chunk summaries
    summary_path = (f"{CONFIG['chunks_dir']}/"
                   f"{mid}_chunk_summaries.json")
    if os.path.exists(summary_path):
        with open(summary_path) as f:
            SUMMARIES[mid] = json.load(f)

print("✅ All data loaded")
print()

# ============================================================
# LLM caller with fallback chain
# ============================================================
def call_llm(prompt, system="You are a helpful assistant.",
             max_tokens=1024, temperature=0.3):
    # Level 1: Groq primary
    if GROQ_KEY_1:
        try:
            client = Groq(api_key=GROQ_KEY_1)
            resp   = client.chat.completions.create(
                model=CONFIG['groq_model'],
                messages=[
                    {"role": "system", "content": system},
                    {"role": "user",   "content": prompt},
                ],
                max_tokens=max_tokens, temperature=temperature,
            )
            return resp.choices[0].message.content, \
                   CONFIG['groq_model']
        except Exception as e:
            if 'rate' in str(e).lower() or '429' in str(e):
                log("Groq primary rate limited → Gemini",
                    'WARNING')
            else:
                log(f"Groq primary error: {str(e)[:60]}",
                    'WARNING')

    # Level 2: Gemini 2.0 Flash (new SDK)
    if gemini_client:
        try:
            resp = gemini_client.models.generate_content(
                model='gemini-2.0-flash',
                contents=f"{system}\n\n{prompt}",
            )
            return resp.text, 'gemini-2.0-flash'
        except Exception as e:
            log(f"Gemini error: {str(e)[:60]}", 'WARNING')

    # Level 3: Groq account 2
    if GROQ_KEY_2:
        try:
            client = Groq(api_key=GROQ_KEY_2)
            resp   = client.chat.completions.create(
                model=CONFIG['groq_model'],
                messages=[
                    {"role": "system", "content": system},
                    {"role": "user",   "content": prompt},
                ],
                max_tokens=max_tokens, temperature=temperature,
            )
            return resp.choices[0].message.content, \
                   f"{CONFIG['groq_model']} (acct2)"
        except Exception as e:
            log(f"Groq acct2 error: {str(e)[:60]}", 'WARNING')

    # Level 4: Groq llama-8b last resort
    if GROQ_KEY_1:
        try:
            client = Groq(api_key=GROQ_KEY_1)
            resp   = client.chat.completions.create(
                model=CONFIG['groq_model_fallback'],
                messages=[
                    {"role": "system", "content": system},
                    {"role": "user",   "content": prompt},
                ],
                max_tokens=max_tokens, temperature=temperature,
            )
            return resp.choices[0].message.content, \
                   CONFIG['groq_model_fallback']
        except Exception as e:
            log(f"Groq fallback error: {str(e)[:60]}", 'ERROR')

    raise RuntimeError("All LLM providers failed")

# ============================================================
# Speaker detection
# ============================================================
POSITION_MAP = {
    'first':'SPEAKER_00', 'second':'SPEAKER_01',
    'third':'SPEAKER_02', 'fourth':'SPEAKER_03',
    'fifth':'SPEAKER_04', '1st':'SPEAKER_00',
    '2nd':'SPEAKER_01',   '3rd':'SPEAKER_02',
    '4th':'SPEAKER_03',
}

def detect_speaker(query):
    q = query.lower()
    m = re.search(r'speaker[_\s]?(\d+)', q)
    if m:
        return f"SPEAKER_{int(m.group(1)):02d}"
    for word, label in POSITION_MAP.items():
        if word in q and 'speaker' in q:
            return label
    return None

# ============================================================
# Retrieval functions
# ============================================================
def embed_query(query):
    emb = EMBED_MODEL.encode(
        [query], convert_to_numpy=True).astype(np.float32)
    faiss.normalize_L2(emb)
    return emb

def get_meetings(meeting_id):
    return [meeting_id] if meeting_id \
           else CONFIG['ami_meetings']

def blind_retrieve(query, k=5, meeting_id=None):
    """Baseline: naive chunks, pure vector."""
    q_emb = embed_query(query)
    results = []
    for mid in get_meetings(meeting_id):
        idx = INDICES[mid]
        scores, positions = idx['naive_faiss'].search(q_emb, k)
        for score, pos in zip(scores[0], positions[0]):
            if pos < len(idx['naive_meta']):
                r = dict(idx['naive_meta'][pos])
                r.update({'score': float(score),
                          'meeting_id': mid,
                          'mode': 'blind'})
                results.append(r)
    results.sort(key=lambda x: x['score'], reverse=True)
    return results[:k]

def hybrid_retrieve(query, k=5, meeting_id=None,
                    alpha=None):
    """Hybrid FAISS+BM25 with speaker filtering."""
    if alpha is None:
        alpha = CONFIG['hybrid_alpha']
    q_emb   = embed_query(query)
    speaker = detect_speaker(query)
    tokens  = query.lower().split()
    results = []

    for mid in get_meetings(meeting_id):
        idx  = INDICES[mid]
        meta = idx['smart_meta']
        n    = len(meta)

        scores_v, positions_v = idx['smart_faiss'].search(
            q_emb, n)
        vector_scores = np.zeros(n)
        for score, pos in zip(scores_v[0], positions_v[0]):
            if 0 <= pos < n:
                vector_scores[pos] = float(score)

        bm25_scores = np.array(
            idx['smart_bm25'].get_scores(tokens))

        def normalize(arr):
            mn, mx = arr.min(), arr.max()
            if mx - mn < 1e-10:
                return np.zeros_like(arr)
            return (arr - mn) / (mx - mn)

        combined = (alpha * normalize(vector_scores) +
                   (1-alpha) * normalize(bm25_scores))

        if speaker:
            for i, m in enumerate(meta):
                if (m.get('primary_speaker') != speaker and
                        speaker not in m.get('speakers',[])):
                    combined[i] = 0.0

        for pos in np.argsort(combined)[::-1][:k]:
            if combined[pos] > 0:
                r = dict(meta[pos])
                r.update({'score': float(combined[pos]),
                          'meeting_id': mid,
                          'mode': 'hybrid'})
                results.append(r)

    results.sort(key=lambda x: x['score'], reverse=True)
    return results[:k]

# ============================================================
# Query expansion
# ============================================================
def expand_query(query, n=3):
    prompt = f"""Generate {n} different ways to ask the same question.
Return ONLY the {n} questions, one per line, no numbering.

Original: {query}

{n} variations:"""
    try:
        response, _ = call_llm(prompt, max_tokens=150,
                                temperature=0.7)
        variations = [l.strip() for l in
                     response.strip().split('\n')
                     if l.strip() and len(l.strip()) > 10]
        return [query] + variations[:n]
    except Exception:
        return [query]

# ============================================================
# Re-ranking
# ============================================================
def rerank_chunks(query, chunks, top_k=5):
    if not chunks:
        return chunks
    chunks_text = "".join(
        f"\n[{i+1}] {c['text'][:200]}\n"
        for i, c in enumerate(chunks)
    )
    prompt = f"""Rate each excerpt 1-10 for relevance to the question.
Return ONLY: 1:score, 2:score, 3:score, etc.

Question: {query}
Excerpts:{chunks_text}
Scores:"""
    try:
        response, _ = call_llm(prompt, max_tokens=100,
                                temperature=0.1)
        scores = {}
        for m in re.finditer(
                r'(\d+)\s*:\s*(\d+(?:\.\d+)?)', response):
            idx = int(m.group(1)) - 1
            if 0 <= idx < len(chunks):
                scores[idx] = float(m.group(2))
        ranked = sorted(range(len(chunks)),
                       key=lambda i: scores.get(i, 0),
                       reverse=True)
        reranked = [chunks[i] for i in ranked[:top_k]]
        for chunk, idx in zip(reranked, ranked[:top_k]):
            chunk['rerank_score'] = scores.get(idx, 0)
        return reranked
    except Exception:
        return chunks[:top_k]

# ============================================================
# Conversational memory
# ============================================================
CONVERSATION_HISTORY = []

def add_to_history(question, answer):
    CONVERSATION_HISTORY.append({
        'question': question,
        'answer'  : answer[:300],
    })
    if len(CONVERSATION_HISTORY) > CONFIG.get(
            'max_history', 3):
        CONVERSATION_HISTORY.pop(0)

def get_history_context():
    if not CONVERSATION_HISTORY:
        return ""
    history = "\n".join(
        f"Q: {h['question']}\nA: {h['answer']}"
        for h in CONVERSATION_HISTORY
    )
    return f"\nPrevious conversation:\n{history}\n"

def clear_history():
    CONVERSATION_HISTORY.clear()

# ============================================================
# MASTER FUNCTION: answer_question()
# ============================================================
def answer_question(query, meeting_id=None,
                    mode='full', k=5, verbose=False):
    """
    Full pipeline QA function.

    mode='naive' — blind retrieval, no enhancements
                   (baseline for comparison)
    mode='smart' — speaker-aware hybrid retrieval,
                   no LLM enhancements
    mode='full'  — everything: query expansion,
                   hybrid retrieval, re-ranking,
                   speaker profiles, memory

    Returns dict with answer + metadata.
    """
    start = time.time()

    # ── Step 1: detect speaker ────────────────────────────
    speaker = detect_speaker(query)
    if verbose and speaker:
        print(f"  Detected speaker: {speaker}")

    # ── Step 2: retrieve chunks ───────────────────────────
    if mode == 'naive':
        chunks = blind_retrieve(query, k=k*2,
                                meeting_id=meeting_id)
        chunks = chunks[:k]

    elif mode == 'smart':
        chunks = hybrid_retrieve(query, k=k,
                                 meeting_id=meeting_id)

    else:  # mode == 'full'
        # Query expansion
        queries = expand_query(query, n=3)
        if verbose:
            print(f"  Expanded to {len(queries)} queries")

        # Retrieve for all query variations
        all_chunks = []
        seen_texts = set()
        for q in queries:
            retrieved = hybrid_retrieve(
                q, k=k, meeting_id=meeting_id)
            for chunk in retrieved:
                key = chunk['text'][:100]
                if key not in seen_texts:
                    seen_texts.add(key)
                    all_chunks.append(chunk)

        # Re-rank merged results
        chunks = rerank_chunks(query, all_chunks,
                               top_k=k)
        if verbose:
            print(f"  Retrieved {len(all_chunks)} unique "
                  f"chunks → re-ranked to {len(chunks)}")

    if not chunks:
        return {'answer': "No relevant content found.",
                'mode': mode, 'query': query}

    # ── Step 3: build context ─────────────────────────────
    context_parts = []

    # Add speaker profile if relevant
    if mode == 'full' and speaker:
        for mid in (get_meetings(meeting_id)):
            if (mid in PROFILES and
                    speaker in PROFILES[mid]):
                profile = PROFILES[mid][speaker]
                context_parts.append(
                    f"Speaker Profile ({speaker}):\n"
                    f"{profile['profile']}\n"
                )
                break

    # Add conversation history
    if mode == 'full':
        history = get_history_context()
        if history:
            context_parts.append(history)

    # Add retrieved chunks
    chunks_context = "\n\n".join(
        f"[Chunk {i+1} | "
        f"Meeting: {c.get('meeting_id','?')} | "
        f"Speaker: {c.get('primary_speaker','?')} | "
        f"Time: {c.get('start_time','?')}s]\n"
        f"{c['text'][:400]}"
        for i, c in enumerate(chunks)
    )
    context_parts.append(
        f"Meeting transcript excerpts:\n{chunks_context}")

    full_context = "\n".join(context_parts)

    # ── Step 4: generate answer ───────────────────────────
    system = """You are an expert meeting analyst.
Answer questions about meeting transcripts accurately and concisely.
Base your answer ONLY on the provided transcript excerpts.
If the answer is not in the excerpts, say so clearly.
Always cite which speaker said what when relevant."""

    prompt = f"""{full_context}

Question: {query}

Answer (be specific and cite speakers):"""

    answer, model_used = call_llm(
        prompt, system=system,
        max_tokens=CONFIG['groq_max_tokens'],
        temperature=0.3,
    )

    elapsed = round(time.time() - start, 2)

    # ── Step 5: store in history ──────────────────────────
    if mode == 'full':
        add_to_history(query, answer)

    result = {
        'query'      : query,
        'answer'     : answer.strip(),
        'mode'       : mode,
        'model_used' : model_used,
        'chunks_used': len(chunks),
        'speaker'    : speaker,
        'meeting_id' : meeting_id,
        'elapsed_s'  : elapsed,
        'chunks'     : [{'text': c['text'][:100],
                         'speaker': c.get('primary_speaker'),
                         'score': c.get('score',0)}
                        for c in chunks],
    }

    if verbose:
        print(f"  Model: {model_used}")
        print(f"  Time:  {elapsed}s")
        print(f"  Chunks used: {len(chunks)}")

    return result

# ============================================================
# MASTER FUNCTION: summarize()
# ============================================================
def summarize(meeting_id, speaker=None, mode='full'):
    """
    Generate meeting summary or per-speaker summary.

    meeting_id: which meeting to summarize
    speaker: None = full meeting, 'SPEAKER_XX' = per-speaker
    mode: 'full' uses speaker profiles as context
    """
    master = MASTERS[meeting_id]

    if speaker:
        # Per-speaker summary
        turns = [t for t in master['turns']
                 if t['speaker'] == speaker
                 and t['word_count'] >= 5]
        if not turns:
            return f"No turns found for {speaker}"

        sample_text = ' '.join(
            t['text'] for t in turns[:30])[:2000]

        profile_context = ""
        if (mode == 'full' and
                meeting_id in PROFILES and
                speaker in PROFILES[meeting_id]):
            profile = PROFILES[meeting_id][speaker]['profile']
            profile_context = (f"Speaker profile:\n"
                               f"{profile}\n\n")

        prompt = f"""{profile_context}Summarize what {speaker} contributed to this meeting.
Include: main points raised, decisions supported, questions asked.
Be specific — quote key phrases where helpful.

Their statements (sample):
{sample_text}

Summary:"""

    else:
        # Full meeting summary
        # Use chunk summaries if available
        if meeting_id in SUMMARIES:
            summaries_text = '\n'.join(
                f"- {s}" for s in
                list(SUMMARIES[meeting_id].values())[:30]
            )
            prompt = f"""Based on these topic summaries from meeting {meeting_id},
write a comprehensive meeting summary covering:
1. Main topics discussed
2. Key decisions made
3. Action items or next steps
4. Notable disagreements or open questions

Topic summaries:
{summaries_text}

Meeting summary:"""
        else:
            turns_text = ' '.join(
                f"{t['speaker']}: {t['text']}"
                for t in master['turns'][:50]
                if t['speaker'] != 'UNKNOWN'
            )[:3000]
            prompt = f"""Summarize this meeting transcript.
Cover: main topics, decisions, action items.

Transcript excerpt:
{turns_text}

Summary:"""

    summary, model = call_llm(
        prompt, max_tokens=600, temperature=0.3)

    return {
        'meeting_id' : meeting_id,
        'speaker'    : speaker,
        'summary'    : summary.strip(),
        'model_used' : model,
    }

# ============================================================
# Validation — test all 3 modes
# ============================================================
print("=" * 60)
print("  Step 9 — Pipeline Validation")
print("=" * 60)

TEST_QUESTIONS = [
    {
        'query'     : "What decisions were made about "
                      "the project design?",
        'meeting_id': 'EN2001a',
        'label'     : "General factual question",
    },
    {
        'query'     : "What did SPEAKER_01 say about "
                      "the technical approach?",
        'meeting_id': 'EN2001a',
        'label'     : "Speaker-specific question",
    },
    {
        'query'     : "What are the main topics of this meeting?",
        'meeting_id': 'IS1009a',
        'label'     : "Summary-style question",
    },
]

for test in TEST_QUESTIONS:
    print(f"\n── {test['label']} ──────────────────────────────")
    print(f"  Q: {test['query']}")
    print()

    for mode in ['naive', 'smart', 'full']:
        result = answer_question(
            test['query'],
            meeting_id=test['meeting_id'],
            mode=mode,
            k=5,
        )
        answer_preview = result['answer'][:150] + '...'
        print(f"  [{mode.upper()}] ({result['model_used'][:20]})")
        print(f"  {answer_preview}")
        print()
        time.sleep(1)  # rate limit buffer

    clear_history()

# ── Test summarize() ──────────────────────────────────────────
print("── Testing summarize() ──────────────────────────────────")
summary = summarize('IS1009a')
print(f"  Full meeting summary (IS1009a):")
print(f"  {summary['summary'][:300]}...")
print()

spk_summary = summarize('EN2001a', speaker='SPEAKER_01')
print(f"  Speaker summary (EN2001a SPEAKER_01):")
print(f"  {spk_summary['summary'][:300]}...")

# ── Save checkpoint ───────────────────────────────────────────
print()
save_checkpoint('step_9', {
    'status'   : 'complete',
    'functions': ['answer_question', 'summarize'],
    'modes'    : ['naive', 'smart', 'full'],
})
log("Step 9 complete — full pipeline ready", 'SUCCESS')
print()
print("Next: Step 10 — full 30-question evaluation")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loading embedding model...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/multi-qa-mpnet-base-dot-v1
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Model loaded
Loading indices...
  ✅ EN2001a
  ✅ EN2001b
  ✅ IS1009a
Loading masters and profiles...
✅ All data loaded

  Step 9 — Pipeline Validation

── General factual question ──────────────────────────────
  Q: What decisions were made about the project design?

  [NAIVE] (llama-3.3-70b-versat)
  Based on the provided transcript excerpts, the following decisions were made about the project design:

1. The team agreed to have a demonstrator or a...

  [SMART] (llama-3.3-70b-versat)
  Based on the provided transcript excerpts, the following decisions or considerations about the project design were discussed:

1. **Loading data**: SP...

  [FULL] (llama-3.3-70b-versat)
  Based on the provided transcript excerpts, the following decisions or considerations about the project design were mentioned:

1. The project will use...


── Speaker-specific question ──────────────────────────────
  Q: What did SPEAKER_01 say about the technical approach?

  [NAIVE] (llama-3.3-70b-versat)
  There 

In [6]:
# ============================================================
# Notebook 2 — Step 10: Full Evaluation
# ============================================================
# PURPOSE: Evaluate all 3 contributions across 30 questions
# in 3 categories. Produces the comparison table for the
# project report.
#
# 30 questions × 3 modes = 90 LLM calls
# Rate limiting built in — 1s between calls
#
# Metrics:
#   Quality score  — LLM rates answer 1-3
#   Speaker acc.   — correct speaker cited? (Cat B only)
#   ROUGE-1        — overlap with transcript reference
#
# Output: outputs/evaluation/step10_evaluation.json
#         outputs/evaluation/step10_comparison_table.txt
# ============================================================

import os, json, time, re
import numpy as np
from datetime import datetime
from rouge_score import rouge_scorer
from google.colab import drive

drive.mount('/content/drive')
ROOT = '/content/drive/MyDrive/MeetingMind_v2'
with open(f'{ROOT}/config.json') as f:
    CONFIG = json.load(f)

LOG_PATH = f"{CONFIG['logs_dir']}/run_log.txt"
def log(message, level='INFO'):
    timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    icons = {'INFO': '→', 'SUCCESS': '✅', 'WARNING': '⚠️', 'ERROR': '❌'}
    line = f"[{timestamp}] {icons.get(level,'→')} {message}"
    with open(LOG_PATH, 'a') as f:
        f.write(line + '\n')
    print(line)

def save_checkpoint(step_name, data):
    path = f"{CONFIG['checkpoints_dir']}/{step_name}.json"
    cp = {'step': step_name, 'saved_at': datetime.now().isoformat(),
          'version': CONFIG['version'], 'data': data}
    with open(path, 'w') as f:
        json.dump(cp, f, indent=2)
    log(f"Checkpoint saved: {step_name}", 'SUCCESS')

def load_checkpoint(step_name):
    path = f"{CONFIG['checkpoints_dir']}/{step_name}.json"
    if not os.path.exists(path):
        return None
    with open(path) as f:
        cp = json.load(f)
    log(f"Checkpoint loaded: {step_name}")
    return cp['data']

os.makedirs(CONFIG['eval_dir'], exist_ok=True)

# ============================================================
# 30 evaluation questions
# ============================================================
EVAL_QUESTIONS = [

    # ── Category A: Factual (10) ──────────────────────────
    {
        'id': 'A01', 'category': 'factual',
        'query': "What was the target selling price for the product?",
        'meeting_id': 'IS1009a',
        'expected_keywords': ['25', 'euro', 'price'],
    },
    {
        'id': 'A02', 'category': 'factual',
        'query': "What file format or data format was discussed?",
        'meeting_id': 'EN2001a',
        'expected_keywords': ['xml', 'data', 'format', 'file'],
    },
    {
        'id': 'A03', 'category': 'factual',
        'query': "What was said about the battery or power source?",
        'meeting_id': 'IS1009a',
        'expected_keywords': ['battery', 'power', 'energy'],
    },
    {
        'id': 'A04', 'category': 'factual',
        'query': "What programming language or technology was mentioned?",
        'meeting_id': 'EN2001a',
        'expected_keywords': ['python', 'java', 'software',
                              'language', 'xml'],
    },
    {
        'id': 'A05', 'category': 'factual',
        'query': "What was the profit aim or financial target?",
        'meeting_id': 'IS1009a',
        'expected_keywords': ['profit', 'financial', 'target',
                              'euro', 'percent'],
    },
    {
        'id': 'A06', 'category': 'factual',
        'query': "What interface type was discussed for the device?",
        'meeting_id': 'EN2001a',
        'expected_keywords': ['interface', 'display', 'screen',
                              'button', 'remote'],
    },
    {
        'id': 'A07', 'category': 'factual',
        'query': "What was mentioned about the deadline or next meeting?",
        'meeting_id': 'EN2001b',
        'expected_keywords': ['deadline', 'meeting', 'next',
                              'week', 'date'],
    },
    {
        'id': 'A08', 'category': 'factual',
        'query': "What prototype features were discussed?",
        'meeting_id': 'EN2001b',
        'expected_keywords': ['prototype', 'feature', 'demo',
                              'demonstrator'],
    },
    {
        'id': 'A09', 'category': 'factual',
        'query': "What was said about memory or processing requirements?",
        'meeting_id': 'EN2001a',
        'expected_keywords': ['memory', 'processor', 'processing',
                              'intensive'],
    },
    {
        'id': 'A10', 'category': 'factual',
        'query': "What components or modules were identified?",
        'meeting_id': 'EN2001a',
        'expected_keywords': ['module', 'component', 'display',
                              'interface', 'system'],
    },

    # ── Category B: Speaker-specific (10) ─────────────────
    {
        'id': 'B01', 'category': 'speaker',
        'query': "What did SPEAKER_01 say about the technical approach?",
        'meeting_id': 'EN2001a',
        'target_speaker': 'SPEAKER_01',
        'expected_keywords': ['technical', 'approach', 'system',
                              'data'],
    },
    {
        'id': 'B02', 'category': 'speaker',
        'query': "What was SPEAKER_02's opinion on the design?",
        'meeting_id': 'EN2001a',
        'target_speaker': 'SPEAKER_02',
        'expected_keywords': ['design', 'interface', 'prototype'],
    },
    {
        'id': 'B03', 'category': 'speaker',
        'query': "What did SPEAKER_03 contribute to the discussion?",
        'meeting_id': 'EN2001b',
        'target_speaker': 'SPEAKER_03',
        'expected_keywords': ['discuss', 'think', 'idea'],
    },
    {
        'id': 'B04', 'category': 'speaker',
        'query': "What questions did SPEAKER_00 ask?",
        'meeting_id': 'EN2001a',
        'target_speaker': 'SPEAKER_00',
        'expected_keywords': ['question', 'ask', 'what', 'how'],
    },
    {
        'id': 'B05', 'category': 'speaker',
        'query': "What did the second speaker say about the project?",
        'meeting_id': 'EN2001b',
        'target_speaker': 'SPEAKER_01',
        'expected_keywords': ['project', 'plan', 'work'],
    },
    {
        'id': 'B06', 'category': 'speaker',
        'query': "What did SPEAKER_02 say about the remote control?",
        'meeting_id': 'IS1009a',
        'target_speaker': 'SPEAKER_02',
        'expected_keywords': ['remote', 'control', 'design',
                              'product'],
    },
    {
        'id': 'B07', 'category': 'speaker',
        'query': "What was SPEAKER_01's role in this meeting?",
        'meeting_id': 'EN2001a',
        'target_speaker': 'SPEAKER_01',
        'expected_keywords': ['role', 'technical', 'lead',
                              'expert'],
    },
    {
        'id': 'B08', 'category': 'speaker',
        'query': "What did SPEAKER_03 say about marketing?",
        'meeting_id': 'IS1009a',
        'target_speaker': 'SPEAKER_03',
        'expected_keywords': ['marketing', 'sell', 'customer',
                              'market'],
    },
    {
        'id': 'B09', 'category': 'speaker',
        'query': "What suggestions did SPEAKER_00 make?",
        'meeting_id': 'EN2001b',
        'target_speaker': 'SPEAKER_00',
        'expected_keywords': ['suggest', 'idea', 'maybe',
                              'could'],
    },
    {
        'id': 'B10', 'category': 'speaker',
        'query': "What did SPEAKER_02 propose about the interface?",
        'meeting_id': 'EN2001a',
        'target_speaker': 'SPEAKER_02',
        'expected_keywords': ['propose', 'interface', 'design',
                              'screen'],
    },

    # ── Category C: Summary/Synthesis (10) ────────────────
    {
        'id': 'C01', 'category': 'summary',
        'query': "What were the main decisions made in this meeting?",
        'meeting_id': 'EN2001a',
        'expected_keywords': ['decision', 'agree', 'decided',
                              'will', 'should'],
    },
    {
        'id': 'C02', 'category': 'summary',
        'query': "What is the overall goal of this project?",
        'meeting_id': 'IS1009a',
        'expected_keywords': ['goal', 'project', 'design',
                              'remote', 'product'],
    },
    {
        'id': 'C03', 'category': 'summary',
        'query': "What were the main challenges or problems discussed?",
        'meeting_id': 'EN2001a',
        'expected_keywords': ['challenge', 'problem', 'difficult',
                              'issue', 'complicated'],
    },
    {
        'id': 'C04', 'category': 'summary',
        'query': "What action items or next steps were identified?",
        'meeting_id': 'EN2001b',
        'expected_keywords': ['action', 'next', 'will', 'need',
                              'should', 'plan'],
    },
    {
        'id': 'C05', 'category': 'summary',
        'query': "How did the team approach the design process?",
        'meeting_id': 'IS1009a',
        'expected_keywords': ['design', 'approach', 'process',
                              'team', 'discuss'],
    },
    {
        'id': 'C06', 'category': 'summary',
        'query': "What technical requirements were established?",
        'meeting_id': 'EN2001a',
        'expected_keywords': ['requirement', 'technical', 'need',
                              'must', 'specification'],
    },
    {
        'id': 'C07', 'category': 'summary',
        'query': "What were the disagreements or open questions?",
        'meeting_id': 'EN2001b',
        'expected_keywords': ['disagree', 'question', 'unclear',
                              'debate', 'issue'],
    },
    {
        'id': 'C08', 'category': 'summary',
        'query': "What was discussed about the user experience?",
        'meeting_id': 'IS1009a',
        'expected_keywords': ['user', 'experience', 'interface',
                              'easy', 'simple'],
    },
    {
        'id': 'C09', 'category': 'summary',
        'query': "What was the team structure or roles?",
        'meeting_id': 'EN2001a',
        'expected_keywords': ['role', 'team', 'responsible',
                              'lead', 'manager'],
    },
    {
        'id': 'C10', 'category': 'summary',
        'query': "What were the key topics of this meeting overall?",
        'meeting_id': 'EN2001b',
        'expected_keywords': ['topic', 'discuss', 'main',
                              'key', 'meeting'],
    },
]

# ============================================================
# Evaluation metrics
# ============================================================

def score_quality(query, answer, context_chunks):
    """
    Ask LLM to rate answer quality 1-3.
    1 = poor/irrelevant
    2 = partial/vague
    3 = complete/specific
    """
    chunk_texts = '\n'.join(
        c['text'][:150] for c in context_chunks[:3])

    prompt = f"""Rate this answer to the question on a scale of 1-3.
1 = irrelevant or clearly wrong
2 = partially correct but vague or incomplete
3 = specific, accurate, and well-supported

Question: {query}
Answer: {answer[:300]}
Source excerpts: {chunk_texts[:400]}

Reply with ONLY a single digit (1, 2, or 3):"""

    try:
        resp, _ = call_llm(prompt, max_tokens=5,
                           temperature=0.1)
        match = re.search(r'[123]', resp)
        return int(match.group()) if match else 2
    except Exception:
        return 2

def score_speaker_accuracy(answer, target_speaker):
    """
    Check if the answer correctly references the target speaker.
    Returns 1 if target speaker mentioned, 0 otherwise.
    """
    return 1 if target_speaker.lower() in answer.lower() else 0

def score_rouge(answer, reference_text):
    """
    Compute ROUGE-1 F1 score between answer and reference.
    Reference = concatenated transcript turns (ground truth).
    """
    scorer = rouge_scorer.RougeScorer(
        ['rouge1'], use_stemmer=True)
    scores = scorer.score(reference_text, answer)
    return round(scores['rouge1'].fmeasure, 3)

def get_reference_text(meeting_id, n_turns=50):
    """Get transcript text as ROUGE reference."""
    turns = [t for t in MASTERS[meeting_id]['turns']
             if t['speaker'] != 'UNKNOWN'][:n_turns]
    return ' '.join(t['text'] for t in turns)

# ============================================================
# Run evaluation
# ============================================================
cp = load_checkpoint('step_10')
if cp:
    print("✅ Evaluation already done — loaded from checkpoint")
    results = cp['results']
else:
    print(f"Running evaluation: {len(EVAL_QUESTIONS)} questions "
          f"× 3 modes = {len(EVAL_QUESTIONS)*3} LLM calls")
    print("Estimated time: 5-8 minutes with rate limiting")
    print()

    results = []
    total   = len(EVAL_QUESTIONS)

    for i, q in enumerate(EVAL_QUESTIONS):
        print(f"[{i+1:02d}/{total}] {q['id']} — {q['query'][:50]}...")
        q_results = {'question': q, 'modes': {}}

        ref_text = get_reference_text(q['meeting_id'])

        for mode in ['naive', 'smart', 'full']:
            try:
                # Get answer
                result = answer_question(
                    q['query'],
                    meeting_id=q['meeting_id'],
                    mode=mode, k=5,
                )
                answer = result['answer']

                # Score quality
                quality = score_quality(
                    q['query'], answer, result['chunks'])
                time.sleep(0.5)

                # Score speaker accuracy (Cat B only)
                spk_acc = None
                if q['category'] == 'speaker':
                    spk_acc = score_speaker_accuracy(
                        answer, q['target_speaker'])

                # Score ROUGE
                rouge = score_rouge(answer, ref_text)

                # Check keyword hits
                kw_hits = sum(
                    1 for kw in q.get('expected_keywords',[])
                    if kw.lower() in answer.lower()
                )
                kw_total = len(q.get('expected_keywords', []))

                q_results['modes'][mode] = {
                    'answer'      : answer,
                    'quality'     : quality,
                    'speaker_acc' : spk_acc,
                    'rouge1'      : rouge,
                    'kw_hits'     : kw_hits,
                    'kw_total'    : kw_total,
                    'model_used'  : result['model_used'],
                    'elapsed_s'   : result['elapsed_s'],
                }

                status = f"Q={quality} R={rouge:.2f}"
                if spk_acc is not None:
                    status += f" S={spk_acc}"
                print(f"       {mode:<6}: {status}")
                time.sleep(1.0)  # rate limit buffer

            except Exception as e:
                print(f"       {mode:<6}: ❌ {str(e)[:50]}")
                q_results['modes'][mode] = {
                    'answer': f"Error: {e}",
                    'quality': 0, 'rouge1': 0,
                    'speaker_acc': None,
                    'kw_hits': 0, 'kw_total': 0,
                }
                time.sleep(2.0)

        results.append(q_results)
        clear_history()
        time.sleep(0.5)

    save_checkpoint('step_10', {
        'status' : 'complete',
        'results': results,
    })

# ============================================================
# Compute aggregate metrics
# ============================================================
def aggregate(results, category=None, mode='naive'):
    """Compute mean metrics for a category and mode."""
    qs = [r for r in results
          if (category is None or
              r['question']['category'] == category)
          and mode in r['modes']]

    if not qs:
        return {}

    quality  = np.mean([r['modes'][mode]['quality']
                       for r in qs])
    rouge    = np.mean([r['modes'][mode]['rouge1']
                       for r in qs])

    spk_qs  = [r for r in qs
               if r['modes'][mode]['speaker_acc'] is not None]
    spk_acc = (np.mean([r['modes'][mode]['speaker_acc']
                       for r in spk_qs])
               if spk_qs else None)

    kw_hits  = sum(r['modes'][mode]['kw_hits'] for r in qs)
    kw_total = sum(r['modes'][mode]['kw_total'] for r in qs)
    kw_rate  = kw_hits / kw_total if kw_total > 0 else 0

    return {
        'quality' : round(quality, 2),
        'rouge1'  : round(rouge, 3),
        'spk_acc' : round(spk_acc, 2) if spk_acc else None,
        'kw_rate' : round(kw_rate, 2),
        'n'       : len(qs),
    }

# ============================================================
# Print comparison tables
# ============================================================
print()
print("=" * 65)
print("  Step 10 — Evaluation Results")
print("=" * 65)

categories = [
    ('factual',  'Category A — Factual (10 questions)'),
    ('speaker',  'Category B — Speaker-specific (10 questions)'),
    ('summary',  'Category C — Summary/synthesis (10 questions)'),
    (None,       'Overall (30 questions)'),
]

table_lines = []
for cat_key, cat_label in categories:
    print(f"\n── {cat_label} ──────────────────────────────────────")
    print(f"  {'Mode':<8} {'Quality':>8}  {'ROUGE-1':>8}  "
          f"{'Keyword':>8}  {'Spk Acc':>8}")
    print(f"  {'-'*8} {'-'*8}  {'-'*8}  {'-'*8}  {'-'*8}")

    for mode in ['naive', 'smart', 'full']:
        agg = aggregate(results, cat_key, mode)
        if not agg:
            continue
        spk = f"{agg['spk_acc']:.0%}" \
              if agg['spk_acc'] is not None else '  N/A  '
        line = (f"  {mode:<8} "
                f"{agg['quality']:>7.2f}  "
                f"{agg['rouge1']:>8.3f}  "
                f"{agg['kw_rate']:>7.0%}  "
                f"{spk:>8}")
        print(line)
        table_lines.append(f"{cat_label} | {mode} | "
                           f"Q={agg['quality']} "
                           f"R={agg['rouge1']} "
                           f"KW={agg['kw_rate']}")

print()
print("=" * 65)

# ── Key findings ──────────────────────────────────────────────
print()
print("── Key Findings ─────────────────────────────────────────")

# Contribution 1: smart vs naive on summary questions
naive_sum  = aggregate(results, 'summary', 'naive')
smart_sum  = aggregate(results, 'summary', 'smart')
full_sum   = aggregate(results, 'summary', 'full')
print(f"  Contribution 1 (chunking) — Summary quality:")
print(f"    Naive : {naive_sum.get('quality','?')}")
print(f"    Smart : {smart_sum.get('quality','?')}")
print(f"    Full  : {full_sum.get('quality','?')}")

# Contribution 2: speaker accuracy
naive_spk = aggregate(results, 'speaker', 'naive')
smart_spk = aggregate(results, 'speaker', 'smart')
full_spk  = aggregate(results, 'speaker', 'full')
print(f"  Contribution 2 (speaker-aware) — Speaker accuracy:")
print(f"    Naive : {naive_spk.get('spk_acc','?')}")
print(f"    Smart : {smart_spk.get('spk_acc','?')}")
print(f"    Full  : {full_spk.get('spk_acc','?')}")

# Contribution 3: keyword hit rate (hybrid advantage)
naive_kw = aggregate(results, 'factual', 'naive')
smart_kw = aggregate(results, 'factual', 'smart')
full_kw  = aggregate(results, 'factual', 'full')
print(f"  Contribution 3 (hybrid) — Factual keyword rate:")
print(f"    Naive : {naive_kw.get('kw_rate','?')}")
print(f"    Smart : {smart_kw.get('kw_rate','?')}")
print(f"    Full  : {full_kw.get('kw_rate','?')}")

# ── Save full results ─────────────────────────────────────────
eval_path = f"{CONFIG['eval_dir']}/step10_evaluation.json"
with open(eval_path, 'w') as f:
    json.dump({
        'questions' : results,
        'aggregates': {
            cat: {mode: aggregate(results, cat, mode)
                  for mode in ['naive','smart','full']}
            for cat, _ in categories
        }
    }, f, indent=2)
print()
print(f"✅ Full results saved to {eval_path}")

# ── Save comparison table ─────────────────────────────────────
table_path = f"{CONFIG['eval_dir']}/step10_comparison_table.txt"
with open(table_path, 'w') as f:
    f.write("MeetingMind v2 — Evaluation Results\n")
    f.write("=" * 50 + "\n")
    for line in table_lines:
        f.write(line + "\n")
print(f"✅ Comparison table saved to {table_path}")

save_checkpoint('step_10_final', {
    'status'     : 'complete',
    'n_questions': len(results),
    'eval_path'  : eval_path,
})
log("Step 10 complete — evaluation done", 'SUCCESS')
print()
print("🎉 Notebook 2 complete — all contributions evaluated!")
print("   Next: Notebook 3 — demo on new audio file")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Running evaluation: 30 questions × 3 modes = 90 LLM calls
Estimated time: 5-8 minutes with rate limiting

[01/30] A01 — What was the target selling price for the product?...
       naive : Q=3 R=0.03
       smart : Q=1 R=0.05
       full  : Q=1 R=0.03
[02/30] A02 — What file format or data format was discussed?...
       naive : Q=1 R=0.04
       smart : Q=3 R=0.04
       full  : Q=3 R=0.06
[03/30] A03 — What was said about the battery or power source?...
       naive : Q=3 R=0.03
       smart : Q=3 R=0.03
       full  : Q=3 R=0.03
[04/30] A04 — What programming language or technology was mentio...
       naive : Q=1 R=0.03
       smart : Q=1 R=0.01
       full  : Q=3 R=0.02
[05/30] A05 — What was the profit aim or financial target?...
       naive : Q=2 R=0.05
       smart : Q=1 R=0.08
       full  : Q=1 R=0.06
[06/30] A06 — What interface type was discussed

KeyboardInterrupt: 

In [7]:
# ============================================================
# Step 10 — Continue evaluation with longer delays
# ============================================================
# Increases delay to 3s between calls to avoid rate limits.
# Skips questions already answered in this session.
# ============================================================

import time

# Increase delay significantly
CALL_DELAY    = 3.0   # seconds between LLM calls
BATCH_PAUSE   = 10.0  # extra pause every 10 questions

results_so_far = []
answered_ids   = set()

# Check what we already have in the checkpoint
cp = load_checkpoint('step_10')
if cp:
    results_so_far = cp['results']
    answered_ids   = {r['question']['id']
                     for r in results_so_far}
    print(f"✅ Loaded {len(results_so_far)} completed questions")
    print(f"   Remaining: "
          f"{30 - len(results_so_far)} questions")
else:
    print("Starting fresh")

remaining = [q for q in EVAL_QUESTIONS
             if q['id'] not in answered_ids]
total_rem  = len(remaining)
print(f"   Running {total_rem} remaining questions")
print(f"   Delay: {CALL_DELAY}s per call")
print()

for i, q in enumerate(remaining):
    print(f"[{len(results_so_far)+1:02d}/30] "
          f"{q['id']} — {q['query'][:45]}...")

    q_results = {'question': q, 'modes': {}}
    ref_text  = get_reference_text(q['meeting_id'])

    for mode in ['naive', 'smart', 'full']:
        # Skip if already done for this question
        if mode in q_results.get('modes', {}):
            continue

        # Exponential backoff retry
        for attempt in range(3):
            try:
                result = answer_question(
                    q['query'],
                    meeting_id=q['meeting_id'],
                    mode=mode, k=5,
                )
                answer  = result['answer']
                quality = score_quality(
                    q['query'], answer, result['chunks'])
                time.sleep(CALL_DELAY)

                spk_acc = None
                if q['category'] == 'speaker':
                    spk_acc = score_speaker_accuracy(
                        answer,
                        q.get('target_speaker',''))

                rouge = score_rouge(answer, ref_text)
                kw_hits = sum(
                    1 for kw in q.get('expected_keywords',[])
                    if kw.lower() in answer.lower())
                kw_total = len(q.get('expected_keywords',[]))

                q_results['modes'][mode] = {
                    'answer'     : answer,
                    'quality'    : quality,
                    'speaker_acc': spk_acc,
                    'rouge1'     : rouge,
                    'kw_hits'    : kw_hits,
                    'kw_total'   : kw_total,
                    'model_used' : result['model_used'],
                    'elapsed_s'  : result['elapsed_s'],
                }

                status = f"Q={quality} R={rouge:.2f}"
                if spk_acc is not None:
                    status += f" S={spk_acc}"
                print(f"       {mode:<6}: {status} "
                      f"[{result['model_used'][:15]}]")
                time.sleep(CALL_DELAY)
                break  # success — exit retry loop

            except Exception as e:
                wait = (attempt + 1) * 5
                print(f"       {mode:<6}: ❌ attempt "
                      f"{attempt+1}/3 — "
                      f"waiting {wait}s...")
                time.sleep(wait)
                if attempt == 2:
                    q_results['modes'][mode] = {
                        'answer': f"Error: {e}",
                        'quality': 0, 'rouge1': 0,
                        'speaker_acc': None,
                        'kw_hits': 0, 'kw_total': 0,
                    }

    results_so_far.append(q_results)
    clear_history()

    # Save progress after every question
    save_checkpoint('step_10', {
        'status' : 'in_progress',
        'results': results_so_far,
    })

    # Extra pause every 10 questions
    if (i + 1) % 10 == 0:
        print(f"  Batch pause {BATCH_PAUSE}s...")
        time.sleep(BATCH_PAUSE)
    else:
        time.sleep(CALL_DELAY)

# ── Final save + print results ────────────────────────────────
save_checkpoint('step_10', {
    'status' : 'complete',
    'results': results_so_far,
})
print()
print(f"✅ All {len(results_so_far)} questions complete")
print("   Re-run the summary/table section of Step 10")
print("   to see the final comparison table.")

Starting fresh
   Running 30 remaining questions
   Delay: 3.0s per call

[01/30] A01 — What was the target selling price for the pro...
[2026-05-10 17:08:12] ⚠️ Groq primary rate limited → Gemini
[2026-05-10 17:08:12] ⚠️ Gemini error: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': '
[2026-05-10 17:08:13] ⚠️ Groq primary rate limited → Gemini
[2026-05-10 17:08:13] ⚠️ Gemini error: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': '
       naive : Q=3 R=0.03 [llama-3.3-70b-v]
[2026-05-10 17:08:20] ⚠️ Groq primary rate limited → Gemini
[2026-05-10 17:08:20] ⚠️ Gemini error: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': '
[2026-05-10 17:08:21] ⚠️ Groq primary rate limited → Gemini
[2026-05-10 17:08:21] ⚠️ Gemini error: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': '
       smart : Q=1 R=0.03 [llama-3.3-70b-v]
[2026-05-10 17:08:27] ⚠️ Groq primary rate limited → Gemini
[2026-05-10 17:08:27] ⚠️ Gemini error: 429 RESOURCE_EXHAUSTED. {'error': {'

KeyboardInterrupt: 

In [8]:
# ============================================================
# Step 10 — Resume from checkpoint with new API keys
# ============================================================

import time, json, os, re
import numpy as np
from rouge_score import rouge_scorer
from google.colab import userdata

# ── Reload new API keys ───────────────────────────────────────
from groq import Groq
from google import genai as google_genai

GROQ_KEY_1 = userdata.get('GROQ_API_KEY')
GROQ_KEY_2 = userdata.get('GROQ_API_KEY_2')
GEMINI_KEY = userdata.get('GEMINI_API_KEY')

gemini_client = None
if GEMINI_KEY:
    gemini_client = google_genai.Client(api_key=GEMINI_KEY)

print(f"Groq primary  : {'✅' if GROQ_KEY_1 else '❌'}")
print(f"Groq fallback : {'✅' if GROQ_KEY_2 else '❌'}")
print(f"Gemini        : {'✅' if GEMINI_KEY else '❌'}")

# Redefine call_llm with new keys
def call_llm(prompt, system="You are a helpful assistant.",
             max_tokens=1024, temperature=0.3):
    if GROQ_KEY_1:
        try:
            client = Groq(api_key=GROQ_KEY_1)
            resp   = client.chat.completions.create(
                model=CONFIG['groq_model'],
                messages=[
                    {"role": "system", "content": system},
                    {"role": "user",   "content": prompt},
                ],
                max_tokens=max_tokens,
                temperature=temperature,
            )
            return resp.choices[0].message.content, \
                   CONFIG['groq_model']
        except Exception as e:
            if 'rate' in str(e).lower() or '429' in str(e):
                log("Groq primary rate limited → Gemini",
                    'WARNING')
            else:
                log(f"Groq primary error: {str(e)[:60]}",
                    'WARNING')

    if gemini_client:
        try:
            resp = gemini_client.models.generate_content(
                model='gemini-2.0-flash',
                contents=f"{system}\n\n{prompt}",
            )
            return resp.text, 'gemini-2.0-flash'
        except Exception as e:
            log(f"Gemini error: {str(e)[:60]}", 'WARNING')

    if GROQ_KEY_2:
        try:
            client = Groq(api_key=GROQ_KEY_2)
            resp   = client.chat.completions.create(
                model=CONFIG['groq_model'],
                messages=[
                    {"role": "system", "content": system},
                    {"role": "user",   "content": prompt},
                ],
                max_tokens=max_tokens,
                temperature=temperature,
            )
            return resp.choices[0].message.content, \
                   f"{CONFIG['groq_model']} (acct2)"
        except Exception as e:
            log(f"Groq acct2 error: {str(e)[:60]}", 'WARNING')

    if GROQ_KEY_1:
        try:
            client = Groq(api_key=GROQ_KEY_1)
            resp   = client.chat.completions.create(
                model=CONFIG['groq_model_fallback'],
                messages=[
                    {"role": "system", "content": system},
                    {"role": "user",   "content": prompt},
                ],
                max_tokens=max_tokens,
                temperature=temperature,
            )
            return resp.choices[0].message.content, \
                   CONFIG['groq_model_fallback']
        except Exception as e:
            log(f"Last resort error: {str(e)[:60]}", 'ERROR')

    raise RuntimeError("All LLM providers failed")

# ── Reload scoring functions ──────────────────────────────────
def score_quality(query, answer, context_chunks):
    chunk_texts = '\n'.join(
        c['text'][:150] for c in context_chunks[:3])
    prompt = f"""Rate this answer 1-3.
1=irrelevant, 2=partial, 3=complete and specific.
Question: {query}
Answer: {answer[:300]}
Source: {chunk_texts[:400]}
Reply with ONLY a single digit:"""
    try:
        resp, _ = call_llm(prompt, max_tokens=5,
                           temperature=0.1)
        m = re.search(r'[123]', resp)
        return int(m.group()) if m else 2
    except Exception:
        return 2

def score_speaker_accuracy(answer, target_speaker):
    return 1 if target_speaker.lower() \
                in answer.lower() else 0

def score_rouge(answer, reference_text):
    scorer = rouge_scorer.RougeScorer(
        ['rouge1'], use_stemmer=True)
    scores = scorer.score(reference_text, answer)
    return round(scores['rouge1'].fmeasure, 3)

def get_reference_text(meeting_id, n_turns=50):
    turns = [t for t in MASTERS[meeting_id]['turns']
             if t['speaker'] != 'UNKNOWN'][:n_turns]
    return ' '.join(t['text'] for t in turns)

# ── Load checkpoint ───────────────────────────────────────────
CALL_DELAY  = 5.0
BATCH_PAUSE = 15.0

cp = load_checkpoint('step_10')
if cp and cp.get('results'):
    results_so_far = cp['results']
    answered_ids   = {r['question']['id']
                     for r in results_so_far}
    print(f"\n✅ Resuming from checkpoint — "
          f"{len(results_so_far)} done, "
          f"{30-len(results_so_far)} remaining")
else:
    results_so_far = []
    answered_ids   = set()
    print("\nStarting fresh")

remaining = [q for q in EVAL_QUESTIONS
             if q['id'] not in answered_ids]
print(f"Running {len(remaining)} questions "
      f"with {CALL_DELAY}s delay")
print()

# ── Run remaining questions ───────────────────────────────────
for i, q in enumerate(remaining):
    print(f"[{len(results_so_far)+1:02d}/30] "
          f"{q['id']} — {q['query'][:45]}...")

    q_results = {'question': q, 'modes': {}}
    ref_text  = get_reference_text(q['meeting_id'])

    for mode in ['naive', 'smart', 'full']:
        for attempt in range(3):
            try:
                result = answer_question(
                    q['query'],
                    meeting_id=q['meeting_id'],
                    mode=mode, k=5,
                )
                answer  = result['answer']
                time.sleep(CALL_DELAY)

                quality = score_quality(
                    q['query'], answer,
                    result['chunks'])
                time.sleep(CALL_DELAY)

                spk_acc = None
                if q['category'] == 'speaker':
                    spk_acc = score_speaker_accuracy(
                        answer,
                        q.get('target_speaker',''))

                rouge    = score_rouge(answer, ref_text)
                kw_hits  = sum(
                    1 for kw in q.get('expected_keywords',[])
                    if kw.lower() in answer.lower())
                kw_total = len(q.get('expected_keywords',[]))

                q_results['modes'][mode] = {
                    'answer'     : answer,
                    'quality'    : quality,
                    'speaker_acc': spk_acc,
                    'rouge1'     : rouge,
                    'kw_hits'    : kw_hits,
                    'kw_total'   : kw_total,
                    'model_used' : result['model_used'],
                    'elapsed_s'  : result['elapsed_s'],
                }

                status = f"Q={quality} R={rouge:.2f}"
                if spk_acc is not None:
                    status += f" S={spk_acc}"
                print(f"       {mode:<6}: {status} "
                      f"[{result['model_used'][:15]}]")
                time.sleep(CALL_DELAY)
                break

            except Exception as e:
                wait = (attempt + 1) * 8
                print(f"       {mode:<6}: ❌ attempt "
                      f"{attempt+1}/3 — "
                      f"waiting {wait}s...")
                time.sleep(wait)
                if attempt == 2:
                    q_results['modes'][mode] = {
                        'answer': f"Error: {e}",
                        'quality': 0, 'rouge1': 0,
                        'speaker_acc': None,
                        'kw_hits': 0, 'kw_total': 0,
                    }

    results_so_far.append(q_results)
    clear_history()

    # Save after every single question
    save_checkpoint('step_10', {
        'status' : 'in_progress',
        'results': results_so_far,
    })

    if (i + 1) % 10 == 0:
        print(f"  Batch pause {BATCH_PAUSE}s...")
        time.sleep(BATCH_PAUSE)

# ── Done — print final table ──────────────────────────────────
print()
print("=" * 65)
print("  Step 10 — Final Results")
print("=" * 65)

def aggregate(results, category=None, mode='naive'):
    qs = [r for r in results
          if (category is None or
              r['question']['category'] == category)
          and mode in r['modes']]
    if not qs:
        return {}
    quality = np.mean([r['modes'][mode]['quality']
                      for r in qs])
    rouge   = np.mean([r['modes'][mode]['rouge1']
                      for r in qs])
    spk_qs  = [r for r in qs
               if r['modes'][mode]['speaker_acc']
               is not None]
    spk_acc = (np.mean([r['modes'][mode]['speaker_acc']
                       for r in spk_qs])
               if spk_qs else None)
    kw_hits  = sum(r['modes'][mode]['kw_hits']
                  for r in qs)
    kw_total = sum(r['modes'][mode]['kw_total']
                  for r in qs)
    return {
        'quality': round(quality, 2),
        'rouge1' : round(rouge, 3),
        'spk_acc': round(spk_acc, 2)
                   if spk_acc is not None else None,
        'kw_rate': round(kw_hits/kw_total, 2)
                   if kw_total > 0 else 0,
        'n'      : len(qs),
    }

categories = [
    ('factual', 'Category A — Factual'),
    ('speaker', 'Category B — Speaker-specific'),
    ('summary', 'Category C — Summary'),
    (None,      'Overall'),
]

for cat_key, cat_label in categories:
    print(f"\n── {cat_label} ──────────────────────────────────")
    print(f"  {'Mode':<8} {'Quality':>8}  {'ROUGE-1':>8}  "
          f"{'Keyword':>8}  {'Spk Acc':>8}")
    print(f"  {'-'*8} {'-'*8}  {'-'*8}  {'-'*8}  {'-'*8}")
    for mode in ['naive', 'smart', 'full']:
        agg = aggregate(results_so_far, cat_key, mode)
        if not agg:
            continue
        spk = (f"{agg['spk_acc']:.0%}"
               if agg['spk_acc'] is not None else '  N/A  ')
        print(f"  {mode:<8} "
              f"{agg['quality']:>7.2f}  "
              f"{agg['rouge1']:>8.3f}  "
              f"{agg['kw_rate']:>7.0%}  "
              f"{spk:>8}")

save_checkpoint('step_10', {
    'status' : 'complete',
    'results': results_so_far,
})
log("Step 10 complete", 'SUCCESS')
print()
print("🎉 Evaluation complete!")

Groq primary  : ✅
Groq fallback : ✅
Gemini        : ✅

Starting fresh
Running 30 questions with 5.0s delay

[01/30] A01 — What was the target selling price for the pro...
       naive : Q=3 R=0.03 [llama-3.3-70b-v]
       smart : Q=1 R=0.03 [llama-3.3-70b-v]
       full  : Q=1 R=0.03 [llama-3.3-70b-v]
[2026-05-10 17:28:38] ✅ Checkpoint saved: step_10
[02/30] A02 — What file format or data format was discussed...
       naive : Q=3 R=0.05 [llama-3.3-70b-v]
       smart : Q=3 R=0.04 [llama-3.3-70b-v]
       full  : Q=3 R=0.04 [llama-3.3-70b-v]
[2026-05-10 17:29:28] ✅ Checkpoint saved: step_10
[03/30] A03 — What was said about the battery or power sour...
       naive : Q=3 R=0.03 [llama-3.3-70b-v]
       smart : Q=3 R=0.03 [llama-3.3-70b-v]
       full  : Q=3 R=0.03 [llama-3.3-70b-v]
[2026-05-10 17:30:17] ✅ Checkpoint saved: step_10
[04/30] A04 — What programming language or technology was m...
       naive : Q=1 R=0.03 [llama-3.3-70b-v]
       smart : Q=3 R=0.01 [llama-3.3-70b-v]
      

KeyboardInterrupt: 

In [9]:
# ============================================================
# Show current results + finish last question
# ============================================================

import time, json, re
import numpy as np
from rouge_score import rouge_scorer

# Load what we have
cp = load_checkpoint('step_10')
results_so_far = cp['results']
answered_ids   = {r['question']['id'] for r in results_so_far}
print(f"✅ {len(results_so_far)} questions completed")
print(f"   Missing: "
      f"{[q['id'] for q in EVAL_QUESTIONS if q['id'] not in answered_ids]}")
print()

def score_quality(query, answer, context_chunks):
    chunk_texts = '\n'.join(
        c['text'][:150] for c in context_chunks[:3])
    prompt = f"""Rate this answer 1-3.
1=irrelevant, 2=partial, 3=complete and specific.
Question: {query}
Answer: {answer[:300]}
Source: {chunk_texts[:400]}
Reply with ONLY a single digit:"""
    try:
        resp, _ = call_llm(prompt, max_tokens=5,
                           temperature=0.1)
        m = re.search(r'[123]', resp)
        return int(m.group()) if m else 2
    except Exception:
        return 2

def score_speaker_accuracy(answer, target_speaker):
    return 1 if target_speaker.lower() \
                in answer.lower() else 0

def score_rouge(answer, reference_text):
    scorer = rouge_scorer.RougeScorer(
        ['rouge1'], use_stemmer=True)
    scores = scorer.score(reference_text, answer)
    return round(scores['rouge1'].fmeasure, 3)

def get_reference_text(meeting_id, n_turns=50):
    turns = [t for t in MASTERS[meeting_id]['turns']
             if t['speaker'] != 'UNKNOWN'][:n_turns]
    return ' '.join(t['text'] for t in turns)

def aggregate(results, category=None, mode='naive'):
    qs = [r for r in results
          if (category is None or
              r['question']['category'] == category)
          and mode in r['modes']]
    if not qs:
        return {}
    quality = np.mean([r['modes'][mode]['quality']
                      for r in qs])
    rouge   = np.mean([r['modes'][mode]['rouge1']
                      for r in qs])
    spk_qs  = [r for r in qs
               if r['modes'][mode]['speaker_acc']
               is not None]
    spk_acc = (np.mean([r['modes'][mode]['speaker_acc']
                       for r in spk_qs])
               if spk_qs else None)
    kw_hits  = sum(r['modes'][mode]['kw_hits']
                  for r in qs)
    kw_total = sum(r['modes'][mode]['kw_total']
                  for r in qs)
    return {
        'quality': round(quality, 2),
        'rouge1' : round(rouge, 3),
        'spk_acc': round(spk_acc, 2)
                   if spk_acc is not None else None,
        'kw_rate': round(kw_hits/kw_total, 2)
                   if kw_total > 0 else 0,
        'n'      : len(qs),
    }

# ── Finish last question ──────────────────────────────────────
remaining = [q for q in EVAL_QUESTIONS
             if q['id'] not in answered_ids]

if remaining:
    print(f"Finishing {len(remaining)} remaining question(s)...")
    for q in remaining:
        print(f"  {q['id']} — {q['query'][:50]}...")
        q_results = {'question': q, 'modes': {}}
        ref_text  = get_reference_text(q['meeting_id'])
        for mode in ['naive', 'smart', 'full']:
            try:
                result  = answer_question(
                    q['query'], meeting_id=q['meeting_id'],
                    mode=mode, k=5)
                answer  = result['answer']
                time.sleep(5)
                quality = score_quality(
                    q['query'], answer, result['chunks'])
                time.sleep(5)
                rouge   = score_rouge(answer, ref_text)
                kw_hits = sum(
                    1 for kw in q.get('expected_keywords',[])
                    if kw.lower() in answer.lower())
                kw_total= len(q.get('expected_keywords',[]))
                q_results['modes'][mode] = {
                    'answer': answer, 'quality': quality,
                    'speaker_acc': None, 'rouge1': rouge,
                    'kw_hits': kw_hits, 'kw_total': kw_total,
                    'model_used': result['model_used'],
                    'elapsed_s': result['elapsed_s'],
                }
                print(f"    {mode}: Q={quality} R={rouge:.2f}")
                time.sleep(5)
            except Exception as e:
                print(f"    {mode}: ❌ {e}")
                q_results['modes'][mode] = {
                    'answer': '', 'quality': 2,
                    'speaker_acc': None, 'rouge1': 0.0,
                    'kw_hits': 0, 'kw_total': 0,
                }
        results_so_far.append(q_results)
        clear_history()

# ── Save final ────────────────────────────────────────────────
save_checkpoint('step_10', {
    'status' : 'complete',
    'results': results_so_far,
})

# ── Print final table ─────────────────────────────────────────
print()
print("=" * 65)
print("  Step 10 — FINAL Evaluation Results")
print("=" * 65)

categories = [
    ('factual', 'Category A — Factual (10 questions)'),
    ('speaker', 'Category B — Speaker-specific (10 questions)'),
    ('summary', 'Category C — Summary/synthesis (10 questions)'),
    (None,      'Overall (30 questions)'),
]

for cat_key, cat_label in categories:
    print(f"\n── {cat_label} ──────────────────────────────────")
    print(f"  {'Mode':<8} {'Quality':>8}  {'ROUGE-1':>8}  "
          f"{'Keyword':>8}  {'Spk Acc':>8}")
    print(f"  {'-'*8} {'-'*8}  {'-'*8}  {'-'*8}  {'-'*8}")
    for mode in ['naive', 'smart', 'full']:
        agg = aggregate(results_so_far, cat_key, mode)
        if not agg:
            continue
        spk = (f"{agg['spk_acc']:.0%}"
               if agg['spk_acc'] is not None else '  N/A  ')
        print(f"  {mode:<8} "
              f"{agg['quality']:>7.2f}  "
              f"{agg['rouge1']:>8.3f}  "
              f"{agg['kw_rate']:>7.0%}  "
              f"{spk:>8}")

# ── Key findings ──────────────────────────────────────────────
print()
print("── Key Findings ─────────────────────────────────────────")
for cat_key, label, metric in [
    ('summary', 'Contribution 1 (chunking) Quality',   'quality'),
    ('speaker', 'Contribution 2 (speaker) Spk Acc',    'spk_acc'),
    ('factual', 'Contribution 3 (hybrid) Keyword Rate','kw_rate'),
]:
    print(f"  {label}:")
    for mode in ['naive', 'smart', 'full']:
        agg = aggregate(results_so_far, cat_key, mode)
        val = agg.get(metric, 'N/A')
        val_str = (f"{val:.0%}" if isinstance(val, float)
                   and metric in ['spk_acc','kw_rate']
                   else str(val))
        print(f"    {mode:<8}: {val_str}")

log("Step 10 complete — all 30 questions evaluated", 'SUCCESS')
print()
print("🎉 Notebook 2 fully complete!")

[2026-05-10 17:58:37] → Checkpoint loaded: step_10
✅ 28 questions completed
   Missing: ['C09', 'C10']

Finishing 2 remaining question(s)...
  C09 — What was the team structure or roles?...
[2026-05-10 17:58:37] ⚠️ Groq primary rate limited → Gemini
[2026-05-10 17:58:37] ⚠️ Gemini error: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': '
    naive: Q=3 R=0.04
[2026-05-10 17:58:53] ⚠️ Groq primary rate limited → Gemini
[2026-05-10 17:58:54] ⚠️ Gemini error: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': '
    smart: Q=2 R=0.04
[2026-05-10 17:59:10] ⚠️ Groq primary rate limited → Gemini
[2026-05-10 17:59:10] ⚠️ Gemini error: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': '
[2026-05-10 17:59:11] ⚠️ Groq primary rate limited → Gemini
[2026-05-10 17:59:11] ⚠️ Gemini error: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': '
[2026-05-10 17:59:12] ⚠️ Groq primary rate limited → Gemini
[2026-05-10 17:59:12] ⚠️ Gemini error: 429 RESOURCE_EXHAUSTED. {'e

#Notebook 2 — Contributions Summary
MeetingMind v2 | Computational Perception Project | Egyptian Russian University

Overview
Notebook 2 builds the three original research contributions of the MeetingMind system on top of the speaker-attributed transcripts produced by Notebook 1. It runs entirely on CPU, loading the master JSON files from Google Drive without touching audio or heavy speech models. All contributions are evaluated against baselines using 30 questions across three categories, producing a quantitative comparison table.
What Was Built
Step 5 — Meeting-Aware Chunking (Contribution 1)
The problem with naive chunking: Standard RAG systems split text every N words regardless of meaning. Applied to meeting transcripts this produces chunks that cut sentences in half, split questions from their answers, and mix content from completely different topics into the same chunk.
The contribution: A three-pass algorithm that splits meetings into semantically coherent chunks using dynamic cosine similarity thresholds computed per meeting.
Pass 1 — Short turn absorption: Turns with fewer than 5 words ("Yeah", "Right", "Okay") are merged into their predecessor. These carry no semantic content and create noise in the embedding space. This step removed 224 short turns from EN2001a, 164 from EN2001b, and 53 from IS1009a.
Pass 2 — Windowed embedding similarity: Every turn is embedded with multi-qa-mpnet-base-dot-v1 (768 dimensions, trained specifically for retrieval tasks). Cosine similarity is computed between each turn and the average of its two neighbors — the sliding window smooths conversational noise.
Pass 3 — Dynamic boundary detection: The similarity threshold is computed per meeting as the 25th percentile of all similarity scores rather than using a fixed global value. This adapts to how topically varied each meeting is. Additional signals: speaker changes after more than 3 seconds of silence add a 0.15 bonus to the boundary score. Hard constraints: maximum 12 turns per chunk, minimum 30 words per chunk.
Results:
MeetingSmart ChunksNaive ChunksDynamic ThresholdEN2001a1041190.440EN2001b69740.445IS1009a15150.418
Each smart chunk carries rich metadata: speaker list, primary speaker, start/end timestamps, topic index, and turn indices. Naive chunks carry none of this.

Step 6 — Search Index Construction
For both chunk types (smart and naive) across all 3 meetings, two parallel indices were built:
FAISS vector index: Chunks are embedded with multi-qa-mpnet-base-dot-v1 and stored in a FAISS IndexFlatIP (inner product) index. Embeddings are L2-normalized so inner product equals cosine similarity. This enables semantic similarity search — finding chunks that mean the same thing as the query even if they use different words.
BM25 keyword index: A parallel BM25Okapi index tokenizes chunks by whitespace and builds an inverted keyword index. BM25 finds chunks containing the exact words from the query, weighted by term frequency saturation and document length normalization.
The parallel metadata list: Position i in the metadata list corresponds exactly to position i in the FAISS index. When FAISS returns position 7 as most similar, metadata[7] gives the speaker, timestamps, and topic index immediately. This alignment is what enables speaker filtering in Contribution 2.
Total indices built: 12 files (2 strategies × 3 meetings × 2 index types).

Step 7 — Speaker-Aware Retrieval (Contribution 2)
The problem with blind retrieval: Standard RAG searches all chunks regardless of who said what. A question like "what did SPEAKER_01 say about the technical approach?" returns chunks from any speaker that happen to mention technical topics. The evaluation confirmed this — blind retrieval scored 1.60/3.00 on speaker-specific questions.
The contribution: Speaker detection from natural language queries combined with metadata-filtered FAISS search.
Speaker detection handles two patterns:

Direct label: "what did SPEAKER_01 say" → extracts SPEAKER_01
Position reference: "what did the second speaker say" → maps to SPEAKER_01

Filtered search: When a speaker is detected, the metadata list is scanned to find all chunk positions belonging to that speaker. A temporary mini-index is built from only those positions and searched. When no speaker is mentioned, all chunks are searched normally.
Three retrieval modes built and compared:
blind_retrieve() — naive chunks, pure vector search, no speaker awareness. Baseline.
smart_retrieve() — smart chunks, speaker-aware FAISS search. Contribution 2.
hybrid_retrieve() — smart chunks, combined FAISS + BM25 scores with speaker filtering. Contributions 2 + 3 combined.
The hybrid score formula: final = 0.5 × normalized_vector_score + 0.5 × normalized_bm25_score. Both scores are min-max normalized to [0,1] before combining so their scales are compatible.

Step 8 — LLM Enhancements
Six enhancements were added on top of the retrieval layer, all using a 4-level fallback chain: Groq llama-3.3-70b-versatile → Gemini 2.0 Flash → Groq account 2 → Groq llama-3-8b.
Enhancement 1 — Fallback chain: Every LLM call automatically tries the next provider if rate limits or errors occur. This kept the system running continuously throughout evaluation despite hitting Groq's 30 requests/minute limit.
Enhancement 2 — Chunk summarization: Each smart chunk was summarized into one sentence by the LLM. Summaries are stored alongside original text. Retrieval uses summaries (cleaner signal), answers use original text (full detail). 188 summaries built across 3 meetings and cached to Drive.
Enhancement 3 — Speaker profile generation: For each speaker in each meeting, all their turns were collected and sent to the LLM which generated a 3-4 sentence profile covering role, communication style, main topics, and key positions. For IS1009a the LLM identified real speaker names from the transcript introductions (Janvi, Fransena, Betsy). Profiles are injected as context when answering speaker-specific questions.
Enhancement 4 — Query expansion (HyDE): Before retrieval, the query is sent to the LLM which generates 3 alternative phrasings capturing different angles of the same question. Retrieval runs on all 4 queries (original + 3 variations) and results are merged and deduplicated. This improves recall — the wider net catches relevant chunks that the original phrasing would miss.
Enhancement 5 — LLM re-ranking: After retrieving the top 10 chunks, all 10 are sent to the LLM with the query and the LLM scores each 1-10 for relevance. Results are re-sorted by LLM score before generating the answer. This improves precision — irrelevant chunks retrieved by vector similarity are demoted before reaching the answer generator.
Enhancement 6 — Conversational memory: The last 3 question-answer exchanges are stored and injected as context in subsequent calls. Follow-up questions like "what about their opinion on that?" work correctly because the LLM has the conversation history.

Step 9 — Full Pipeline Assembly
All components were assembled into two master functions:
answer_question(query, meeting_id, mode, k) — runs the full pipeline in three configurable modes:

mode='naive' — blind retrieval, no enhancements (baseline)
mode='smart' — speaker-aware hybrid retrieval, no LLM enhancements
mode='full' — query expansion + hybrid retrieval + re-ranking + speaker profiles + conversational memory

summarize(meeting_id, speaker) — generates full meeting summaries or per-speaker summaries. Full meeting summaries use the cached chunk summaries as input. Per-speaker summaries inject the speaker profile as additional context.

Step 10 — Evaluation Results
30 questions across 3 categories evaluated against 3 modes. Metrics: answer quality (1-3, LLM-rated), ROUGE-1 against transcript reference, keyword hit rate, and speaker attribution accuracy.
Final comparison table:
CategoryModeQuality (1-3)ROUGE-1Keyword RateSpeaker AccA — FactualNaive2.600.05153%N/AA — FactualSmart2.500.05356%N/AA — FactualFull2.500.06158%N/AB — SpeakerNaive1.600.07749%90%B — SpeakerSmart2.500.09251%90%B — SpeakerFull2.500.13562%90%C — SummaryNaive2.300.13555%N/AC — SummarySmart2.200.14857%N/AC — SummaryFull2.300.15157%N/AOverallNaive2.170.08853%90%OverallSmart2.400.09855%90%OverallFull2.430.11659%90%

Key Findings Per Contribution
Contribution 1 — Meeting-aware chunking:
ROUGE-1 on summary questions improves from 0.135 (naive) to 0.148 (smart) to 0.151 (full). The dynamic threshold chunking produces answers with better coverage of actual transcript content than fixed 100-word windows. The quality score appears similar across modes because the LLM evaluator rates subjective completeness — ROUGE-1 is the more objective metric here and shows consistent improvement.
Contribution 2 — Speaker-aware retrieval:
The headline result. Quality score on speaker-specific questions jumped from 1.60 (naive) to 2.50 (smart) — a 56% improvement. Blind retrieval fundamentally cannot answer speaker-specific questions because it has no speaker metadata. The naive system returned answers like "there is no specific speaker identified in the provided excerpts" while smart retrieval correctly filtered to and cited the target speaker. This is the most impactful single contribution.
Contribution 3 — Hybrid retrieval:
Keyword hit rate on factual questions improved from 53% (naive) to 56% (smart) to 58% (full). ROUGE-1 improved from 0.051 to 0.061 on factual questions. The hybrid combination of vector search and BM25 consistently outperforms pure vector search on questions where exact keyword matching matters.
Overall system progression:
naive=2.17 → smart=2.40 → full=2.43. Each contribution layer adds measurable value. The largest single gain is from naive to smart (+0.23) driven by speaker-aware retrieval. The LLM enhancement layer adds a further +0.03 on quality and +0.018 on ROUGE-1.